# KIVI: 2-Bit KV-Cache Quantization for LLaMA (7B & 13B)

**Course:** CMSC 723 — Graduate NLP  
**Author:** Kiyana Amirian  
**Team:** Kiyana Amirian, Helia Mohammadpour, Nick Milionis, Hengyuan Liu, Saketh Challagundla

---

## Overview

This notebook implements and evaluates **KIVI** — a training-free, 2-bit KV-cache quantization scheme for large language models, applied to LLaMA-2 7B and 13B. KIVI significantly reduces KV-cache memory footprint while maintaining generation quality across multiple benchmarks.

### Key idea

Instead of storing full-precision (FP16) key-value tensors in the attention cache, KIVI:
1. Quantizes tokens to **2 bits** (per-channel for Keys, per-token for Values) using group quantization
2. Maintains a small **residual buffer** of recent full-precision tokens to preserve local context
3. Dequantizes on-the-fly during attention computation — no retraining required

### Structure

| Section | Description |
|---------|-------------|
| 1. Quantization primitives | Per-token and per-channel 2-bit quantization with group-size calibration |
| 2. KIVI cache manager | `KIVICache` class: quantized storage + residual buffer + dequantization |
| 3. LLaMA attention replacement | Drop-in KIVI attention module; `replace_llama_attention_with_kivi` |
| 4. Memory profiling | Theoretical KV-cache MB and empirical GPU peak memory |
| 5. CNN/DailyMail evaluation | Summarization benchmark — ROUGE, BERTScore, token match |
| 6. GSM8K evaluation | Math reasoning benchmark — exact match accuracy (7B and 13B) |
| 7. CoQA evaluation | Conversational QA benchmark |

### Results summary

| Model | Benchmark | Baseline | KIVI | Memory reduction |
|-------|-----------|----------|------|------------------|
| LLaMA-2 7B | CNN/DM (ROUGE-L) | — | — | ~8× theoretical |
| LLaMA-2 13B | GSM8K (EM) | — | — | ~8× theoretical |
| LLaMA-2 7B | CoQA | — | — | ~8× theoretical |

> **Note:** Results are stored in `results/`. The model weights require a HuggingFace token with LLaMA-2 access. GPU recommended (A100/H100 for 13B).


In [ ]:
import torch
import math
def quantize_per_token(X, num_bits=2, group_size=32):
    """
    Quantize along token dimension (for value cache)
    X: shape [num_tokens, hidden_dim] x: [T, D]evaluate_on_cnn_dm_from_texts
    """
    # Group along hidden dimension
    num_groups = X.shape[1] // group_size
    X_grouped = X[:, :num_groups * group_size].reshape(
        X.shape[0], num_groups, group_size
    ) # getting [T, number of group, group_size]



    # Calculate per-token statistics
    z_X = X_grouped.min(dim= -1, keepdim=True)[0]  # zero-point # you remove the 32 dimension by reducing and keep singleton dimension
    max_X = X_grouped.max(dim= -1, keepdim=True)[0]
    s_X = (max_X - z_X) / (2**num_bits - 1)  # scaling factor

    # Quantize
    X_quant = torch.round((X_grouped - z_X) / s_X)

    return X_quant, s_X, z_X

def quantize_per_channel(X, num_bits=2, group_size=32):
    """
    Quantize along channel dimension (for key cache)
    X: shape [num_tokens, hidden_dim] x: [T, D]
    """
    # Group along token dimension
    num_groups = X.shape[0] // group_size
    X_grouped = X[:num_groups * group_size, :].reshape(
        num_groups, group_size, X.shape[1]
    ) # [number of group, group_size (32 tokens), D(the entire hidden dimension)]

    # Calculate per-channel statistics
    z_X = X_grouped.min(dim=1, keepdim=True)[0]  # zero-point # z_x shape is [num_groups, 1, D], meaning that for group 0--> z_x[0] is a vector of lenght D
    max_X = X_grouped.max(dim=1, keepdim=True)[0]
    s_X = (max_X - z_X) / (2**num_bits - 1)  # scaling factor

    # Quantize
    X_quant = torch.round((X_grouped - z_X) / s_X)

    return X_quant, s_X, z_X


def dequantize(X_quant, s_X, z_X):
    """
    tize back to float"""
    return X_quant * s_X + z_X

## 2. KIVI Cache Manager


In [ ]:
class KIVICache:
    def __init__(self, num_bits=2, group_size=32, residual_length=128):
        self.num_bits = num_bits
        self.group_size = group_size  # G in the paper
        self.residual_length = residual_length  # R in the paper

        # Grouped parts (quantized)
        self.key_grouped_quant = None
        self.key_grouped_scale = None
        self.key_grouped_zero = None

        self.value_grouped_quant = None
        self.value_grouped_scale = None
        self.value_grouped_zero = None

        # Residual parts (full precision)
        self.key_residual = None  # Leftover tokens that don't form complete group
        self.value_residual = None  # Last R tokens (sliding window)

    def prefill(self, key_states, value_states):
        """Initialize cache during prefill phase"""
        num_tokens = key_states.shape[0]

        # Ensure R is divisible by G (required by paper)
        assert self.residual_length % self.group_size == 0, \
            "R must be divisible by G"

        # === KEY CACHE: r = l % R (from KeyQuant function) === or # === KEY CACHE: r = l % G (true KeyQuant rule, PDF typo corrected) === for now Im using G instead of R
        #  Note: The paper has a typo here; it should be r = l % G or at least I think this way.
        r = num_tokens % self.group_size  # ← Use G, not R
        num_grouped_tokens = num_tokens - r

        if num_grouped_tokens > 0:
            key_grouped = key_states[:num_grouped_tokens]
            self.key_grouped_quant, self.key_grouped_scale, self.key_grouped_zero = \
                quantize_per_channel(key_grouped, self.num_bits, self.group_size)
            self.key_residual = key_states[num_grouped_tokens:]
        else:
            self.key_grouped_quant = None
            self.key_grouped_scale = None
            self.key_grouped_zero = None
            self.key_residual = key_states

        # === VALUE CACHE: XVr = XV[lprompt - R:] ===
        if num_tokens > self.residual_length:
            value_grouped = value_states[:-self.residual_length]
            self.value_grouped_quant, self.value_grouped_scale, self.value_grouped_zero = \
                quantize_per_token(value_grouped, self.num_bits, self.group_size)
            self.value_residual = value_states[-self.residual_length:]
        else:
            self.value_grouped_quant = None
            self.value_grouped_scale = None
            self.value_grouped_zero = None
            self.value_residual = value_states





    def update(self, new_key, new_value):
        """Update cache during decoding"""

        # ====================================================
        # KEY CACHE UPDATE (matches Algorithm 1 exactly)
        # ====================================================
        if self.key_residual is None:
            self.key_residual = new_key
        else:
            self.key_residual = torch.cat([self.key_residual, new_key], dim=0)

        # Flush when residual reaches EXACTLY R tokens
        if self.key_residual.shape[0] == self.residual_length:

            # Quantize whole R-token block
            key_quant, key_scale, key_zero = quantize_per_channel(
                self.key_residual, self.num_bits, self.group_size
            )

            # Append to grouped key cache
            if self.key_grouped_quant is not None:
                self.key_grouped_quant = torch.cat([self.key_grouped_quant, key_quant], dim=0)
                self.key_grouped_scale = torch.cat([self.key_grouped_scale, key_scale], dim=0)
                self.key_grouped_zero  = torch.cat([self.key_grouped_zero,  key_zero],  dim=0)
            else:
                self.key_grouped_quant = key_quant
                self.key_grouped_scale = key_scale
                self.key_grouped_zero  = key_zero

            # Reset residual
            self.key_residual = None

        # ====================================================
        # VALUE CACHE UPDATE (correct as you wrote)
        # ====================================================
        if self.value_residual is None:
            self.value_residual = new_value
        else:
            self.value_residual = torch.cat([self.value_residual, new_value], dim=0)

        # If residual exceeds R, quantize oldest part
        if self.value_residual.shape[0] > self.residual_length:
            num_to_quantize = self.value_residual.shape[0] - self.residual_length
            to_quantize = self.value_residual[:num_to_quantize]

            value_quant, value_scale, value_zero = quantize_per_token(
                to_quantize, self.num_bits, self.group_size
            )

            if self.value_grouped_quant is not None:
                self.value_grouped_quant = torch.cat([self.value_grouped_quant, value_quant], dim=0)
                self.value_grouped_scale = torch.cat([self.value_grouped_scale, value_scale], dim=0)
                self.value_grouped_zero  = torch.cat([self.value_grouped_zero,  value_zero],  dim=0)
            else:
                self.value_grouped_quant = value_quant
                self.value_grouped_scale = value_scale
                self.value_grouped_zero  = value_zero

            self.value_residual = self.value_residual[-self.residual_length:]

    # ========== INTERNAL DEQUANTIZATION METHODS ==========
    def _dequantize_per_channel(self, X_quant, s_X, z_X):
        """
        Dequantize key cache
        X_quant: [num_groups, group_size, D]
        s_X: [num_groups, 1, D]
        z_X: [num_groups, 1, D]
        Returns: [num_groups, group_size, D] (still grouped, reshape outside)
        """
        return X_quant * s_X + z_X

    def _dequantize_per_token(self, X_quant, s_X, z_X):
        """
        Dequantize value cache (values are dequantized across hidden dimension, not across tokens)
        X_quant: [T, num_groups, group_size] T is number of tokens in grouped value cache
        s_X: [T, num_groups, 1]
        z_X: [T, num_groups, 1]
        Returns: [T, num_groups, group_size] (still grouped, reshape outside)
        """
        return X_quant * s_X + z_X

    # ---------- PUBLIC ACCESSORS ----------
    def get_keys(self):
        """
        Reconstruct full key cache: [T_total, D]
        Concatenates:
        - dequantized grouped keys: Q(XKg) -> float
        - residual (unquantized) keys: XKr
        """
        parts = []

        if self.key_grouped_quant is not None:
            # dequant: [Ng, G, D]
            key_grouped = self._dequantize_per_channel(
                self.key_grouped_quant,
                self.key_grouped_scale,
                self.key_grouped_zero,
            )
            # reshape to [Ng*G, D]
            Ng, G, D = key_grouped.shape
            key_grouped = key_grouped.reshape(Ng * G, D)
            parts.append(key_grouped)

        if self.key_residual is not None:
            parts.append(self.key_residual)   # [Tr, D]

        if not parts:
            return None

        return torch.cat(parts, dim=0)  # [T_total, D]

    def get_values(self):
        """
        Reconstruct full value cache: [T_total, D]
        Concatenates:
        - dequantized grouped values: Q(XVg)
        - residual (unquantized) values: XVr (last R tokens)
        """
        parts = []

        if self.value_grouped_quant is not None:
            # dequant: [Tg, Ngv, G]
            value_grouped = self._dequantize_per_token(
                self.value_grouped_quant,
                self.value_grouped_scale,
                self.value_grouped_zero,
            )
            Tg, Ngv, G = value_grouped.shape
            D = Ngv * G
            value_grouped = value_grouped.reshape(Tg, D)
            parts.append(value_grouped)

        if self.value_residual is not None:
            parts.append(self.value_residual)

        if not parts:
            return None

        return torch.cat(parts, dim=0)  # [T_total, D]

    ####### Utility methods #######

    def key_lengths(self):
        """
            Returns: (grouped_len, residual_len)
            grouped_len: number of tokens in quantized key cache
            residual_len: number of tokens in residual key cache
            """
        grouped_len = 0
        if self.key_grouped_quant is not None:
            Ng, G, D = self.key_grouped_quant.shape
            grouped_len = Ng * G

        residual_len = 0
        if self.key_residual is not None:
            residual_len = self.key_residual.shape[0]

        return grouped_len, residual_len


    def value_lengths(self):
        """
            Returns: (grouped_len, residual_len)
            grouped_len: number of tokens in quantized value cache
            residual_len: number of tokens in residual value cache
            """
        grouped_len = 0
        if self.value_grouped_quant is not None:
            Tg, Ngv, G = self.value_grouped_quant.shape
            grouped_len = Tg  # Tg tokens

        residual_len = 0
        if self.value_residual is not None:
            residual_len = self.value_residual.shape[0]

        return grouped_len, residual_len

    def get_memory_stats(self):
        """Calculate memory usage statistics

        Note: This calculates theoretical memory if quantized values were bit-packed.
        In practice, PyTorch stores quantized tensors as full-precision, so actual
        memory usage will be higher until bit-packing is implemented.
        """
        stats = {
            'key_residual_bytes': 0,
            'value_residual_bytes': 0,
            'key_quant_bytes': 0,
            'value_quant_bytes': 0,
            'key_metadata_bytes': 0,
            'value_metadata_bytes': 0,
        }

        if self.key_residual is not None:
            stats['key_residual_bytes'] = (
                self.key_residual.element_size() * self.key_residual.nelement()
            )
        if self.value_residual is not None:
            stats['value_residual_bytes'] = (
                self.value_residual.element_size() * self.value_residual.nelement()
            )

        if self.key_grouped_quant is not None:
            # Calculate theoretical memory for quantized values (bit-packed)
            num_elements = self.key_grouped_quant.nelement()
            stats['key_quant_bytes'] = (num_elements * self.num_bits) / 8  # bits to bytes

            # Metadata (scale/zero) are stored as full precision
            stats['key_metadata_bytes'] = (
                self.key_grouped_scale.element_size() * self.key_grouped_scale.nelement() +
                self.key_grouped_zero.element_size() * self.key_grouped_zero.nelement()
            )

        if self.value_grouped_quant is not None:
            # Calculate theoretical memory for quantized values (bit-packed)
            num_elements = self.value_grouped_quant.nelement()
            stats['value_quant_bytes'] = (num_elements * self.num_bits) / 8  # bits to bytes

            # Metadata (scale/zero) are stored as full precision
            stats['value_metadata_bytes'] = (
                self.value_grouped_scale.element_size() * self.value_grouped_scale.nelement() +
                self.value_grouped_zero.element_size() * self.value_grouped_zero.nelement()
            )

        stats['total_bytes'] = sum(stats.values())
        stats['total_mb'] = stats['total_bytes'] / (1024 * 1024)

        return stats

    def get_memory_stats_gpu_memory(self):
        """Calculate memory usage statistics"""
        stats = {
            'key_residual_bytes': 0,
            'value_residual_bytes': 0,
            'key_quant_bytes': 0,
            'value_quant_bytes': 0,
            'key_metadata_bytes': 0,
            'value_metadata_bytes': 0,
        }

        if self.key_residual is not None:
            stats['key_residual_bytes'] = (
                self.key_residual.element_size() * self.key_residual.nelement()
            )
        if self.value_residual is not None:
            stats['value_residual_bytes'] = (
                self.value_residual.element_size() * self.value_residual.nelement()
            )

        if self.key_grouped_quant is not None:
            stats['key_quant_bytes'] = (
                self.key_grouped_quant.element_size() * self.key_grouped_quant.nelement()
            )
            stats['key_metadata_bytes'] = (
                self.key_grouped_scale.element_size() * self.key_grouped_scale.nelement() +
                self.key_grouped_zero.element_size() * self.key_grouped_zero.nelement()
            )

        if self.value_grouped_quant is not None:
            stats['value_quant_bytes'] = (
                self.value_grouped_quant.element_size() * self.value_grouped_quant.nelement()
            )
            stats['value_metadata_bytes'] = (
                self.value_grouped_scale.element_size() * self.value_grouped_scale.nelement() +
                self.value_grouped_zero.element_size() * self.value_grouped_zero.nelement()
            )

        stats['total_bytes'] = sum(stats.values())
        stats['total_mb'] = stats['total_bytes'] / (1024 * 1024)

        return stats

    # ========== ATTENTION COMPUTATION (OPTIONAL) ==========
    # Add this to your KIVICache class

    def compute_attention(self, query, use_split=False, attention_mask=None):
        """
        Compute attention following Algorithm 1

        Args:
            query: [1, D] or [batch, D]
            use_split: If True, use optimized split attention (avoids full dequantization)
                    If False, use simple method (full dequantization)

        Returns:
            output: [1, D] or [batch, D]
        """
        if use_split:
            return self._compute_attention_split(query, attention_mask=attention_mask)
        else:
            return self._compute_attention_simple(query, attention_mask=attention_mask)

    def _compute_attention_simple(self, query, attention_mask=None):
        """
        Simple version of attention:
        - fully dequantize
        - apply causal mask
        - apply attention mask (optional)
        """
        keys = self.get_keys()      # [T, D]
        values = self.get_values()  # [T, D]
        T = keys.shape[0]
        D = keys.shape[1]

        # Step 1: raw scores
        scores = torch.matmul(query, keys.T) / math.sqrt(D)   # [1, T]

        # -------------------------------
        # Step 3: Optional padding mask
        # -------------------------------
        if attention_mask is not None:
            scores = scores + attention_mask  # shape must be [1, T]

        # Step 4: softmax
        attn_weights = torch.softmax(scores, dim=-1)

        # Step 5: weighted sum
        output = torch.matmul(attn_weights, values)  # [1, D]

        return output

    def _compute_attention_split(self, query, attention_mask=None):
        """
        Split attention following Algorithm 1 (Equation 3)
        More efficient: computes attention on grouped and residual parts separately

        IMPORTANT: The split is based on VALUE cache structure, not key cache!
        This is because we split attention weights to apply to values.
        """
        # ========== STEP 1: Compute attention scores for ALL keys ==========
        scores_grouped = None
        scores_residual = None
        scale = math.sqrt(query.size(-1))

        # Compute scores for grouped keys (quantized)
        if self.key_grouped_quant is not None:
            # Dequantize grouped keys
            keys_grouped = self._dequantize_per_channel(
                self.key_grouped_quant,
                self.key_grouped_scale,
                self.key_grouped_zero
            )
            # Reshape: [Ng, G, D] -> [Ng*G, D]
            Ng, G, D = keys_grouped.shape
            keys_grouped = keys_grouped.reshape(Ng * G, D)

            # Compute scores: [batch, Ng*G]
            scores_grouped = torch.matmul(query, keys_grouped.T) / scale

        # Compute scores for residual keys (full precision)
        if self.key_residual is not None:
            # [batch, Tr]
            scores_residual = torch.matmul(query, self.key_residual.T) / scale

        # Concatenate all scores
        if scores_grouped is not None and scores_residual is not None:
            scores = torch.cat([scores_grouped, scores_residual], dim=-1)
        elif scores_grouped is not None:
            scores = scores_grouped
        elif scores_residual is not None:
            scores = scores_residual
        else:
            return None

        scores = scores

         # -------------------------------
        # STEP 3: Apply user attention_mask
        # -------------------------------
        if attention_mask is not None:
            scores = scores + attention_mask   # shape [1, T]

        # -------------------------------
        # STEP 4: Softmax
        # -------------------------------
        attn_weights = torch.softmax(scores, dim=-1)

        # ========== STEP 5: Split weights by VALUE structure ==========
        value_grouped_len = 0
        if self.value_grouped_quant is not None:
            Tg, Ngv, G = self.value_grouped_quant.shape
            value_grouped_len = Tg  # Number of tokens in grouped values

        value_residual_len = 0
        if self.value_residual is not None:
            value_residual_len = self.value_residual.shape[0]

        total_value_len = value_grouped_len + value_residual_len

        # Check that total matches
        if attn_weights.shape[-1] != total_value_len:
            raise RuntimeError(
                f"Attention weights length ({attn_weights.shape[-1]}) doesn't match "
                f"total value cache length ({total_value_len}). "
                f"This should not happen - key and value caches are misaligned!"
            )

        # Split weights according to value cache structure
        if value_grouped_len > 0 and value_residual_len > 0:
            attn_grouped = attn_weights[:, :value_grouped_len]      # [batch, value_grouped_len]
            attn_residual = attn_weights[:, value_grouped_len:]     # [batch, value_residual_len]
        elif value_grouped_len > 0:
            attn_grouped = attn_weights
            attn_residual = None
        else:
            attn_grouped = None
            attn_residual = attn_weights

        # ========== STEP 4: Compute output ==========
        output = None

        # Grouped values contribution: Ag @ Q(XVg)
        if attn_grouped is not None and self.value_grouped_quant is not None:
            # Dequantize grouped values
            values_grouped = self._dequantize_per_token(
                self.value_grouped_quant,
                self.value_grouped_scale,
                self.value_grouped_zero
            )
            # Reshape: [Tg, Ngv, G] -> [Tg, D]
            Tg, Ngv, G = values_grouped.shape
            D = Ngv * G
            values_grouped = values_grouped.reshape(Tg, D)

            # Compute: [batch, Tg] @ [Tg, D] -> [batch, D]
            output = torch.matmul(attn_grouped, values_grouped)

        # Residual values contribution: Ar @ XVr
        if attn_residual is not None and self.value_residual is not None:
            # Compute: [batch, Tr] @ [Tr, D] -> [batch, D]
            residual_output = torch.matmul(attn_residual, self.value_residual)

            if output is not None:
                output = output + residual_output
            else:
                output = residual_output

        return output

# ========== HELPER FUNCTIONS (OUTSIDE CLASS) ==========

def get_kivi_memory_stats(model):
    """
    Calculate memory usage of KIVI caches in a model

    Args:
        model: Model with KIVI-enabled attention layers

    Returns:
        dict with total memory statistics
    """
    total_bytes = 0

    for name, module in model.named_modules():
        # Check if this is a KIVI-enabled attention module
        if hasattr(module, 'kivi_cache') and module.kivi_cache is not None:
            for cache in module.kivi_cache:
                if hasattr(cache, 'get_memory_stats'):
                    stats = cache.get_memory_stats()
                    total_bytes += stats.get('total_bytes', 0)

    return {
        'total_bytes': total_bytes,
        'total_mb': total_bytes / (1024 * 1024),
        'total_gb': total_bytes / (1024 * 1024 * 1024)
    }

### Unit Tests


In [ ]:
import torch

# Create cache
cache = KIVICache(num_bits=2, group_size=32, residual_length=128)

# Prefill
keys = torch.randn(550, 4096)
values = torch.randn(550, 4096)
cache.prefill(keys, values)

# Query
query = torch.randn(1, 4096)

# Method 1: Simple (default)
output_simple = cache.compute_attention(query, use_split=False)
print(f"Simple output shape: {output_simple.shape}")

# Method 2: Split (optimized)
output_split = cache.compute_attention(query, use_split=True)
print(f"Split output shape: {output_split.shape}")

# Verify they produce similar results
diff = torch.abs(output_simple - output_split).mean()
print(f"Difference between methods: {diff.item():.6f}")

In [ ]:
for _ in range(10):
    new_k = torch.randn(1, 4096)
    new_v = torch.randn(1, 4096)
    cache.update(new_k, new_v)
    out_s = cache.compute_attention(query, use_split=False)
    out_sp = cache.compute_attention(query, use_split=True)
    print((out_s - out_sp).abs().max().item())


In [ ]:
# Create cache
cache = KIVICache()

# Use it
keys = torch.randn(550, 4096)
values = torch.randn(550, 4096)
cache.prefill(keys, values)

# Get memory stats
stats = cache.get_memory_stats()
print(f"Memory usage: {stats['total_mb']:.2f} MB")


# using on Lama
## Llama attention with KiVi

In [ ]:
import torch
import torch.nn as nn
from transformers.models.llama.modeling_llama import LlamaAttention, LlamaRotaryEmbedding
from typing import Optional, Tuple
import math

class LlamaAttentionWithKIVI(nn.Module):
    """
    Modified LlamaAttention that uses KIVI cache
    """

    def __init__(self, config, layer_idx: Optional[int] = None):
        super().__init__()
        self.config = config
        self.layer_idx = layer_idx

        self.attention_dropout = config.attention_dropout
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_key_value_heads = config.num_key_value_heads
        self.num_key_value_groups = self.num_heads // self.num_key_value_heads
        self.max_position_embeddings = config.max_position_embeddings
        self.rope_theta = config.rope_theta

        if (self.head_dim * self.num_heads) != self.hidden_size:
            raise ValueError(
                f"hidden_size must be divisible by num_heads (got `hidden_size`: {self.hidden_size}"
                f" and `num_heads`: {self.num_heads})."
            )

        # Linear projections
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=config.attention_bias)
        self.k_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=config.attention_bias)
        self.v_proj = nn.Linear(self.hidden_size, self.num_key_value_heads * self.head_dim, bias=config.attention_bias)
        self.o_proj = nn.Linear(self.hidden_size, self.hidden_size, bias=config.attention_bias)

        # Rotary embeddings
        self.rotary_emb = LlamaRotaryEmbedding(config=config)

        # KIVI cache settings aka hyperparameters
        self.use_kivi = True
        self.kivi_num_bits = 2
        self.kivi_group_size = 32
        self.kivi_residual_length = 128
        self.kivi_cache = None  # Will be initialized per head

    def _init_kivi_cache(self):
        """Initialize KIVI cache for each KV head"""
        if self.kivi_cache is None:
            self.kivi_cache = [
                KIVICache(
                    num_bits=self.kivi_num_bits,
                    group_size=self.kivi_group_size,
                    residual_length=self.kivi_residual_length
                )
                for _ in range(self.num_key_value_heads)
            ]
    def reset_kivi_cache(self):
        self.kivi_cache = None

    def forward(


        self,
        hidden_states: torch.Tensor,
        position_embeddings: Tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor] = None,
        past_key_values: Optional["Cache"] = None,
        cache_position: Optional[torch.LongTensor] = None,
        **kwargs,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor], Optional[Tuple[torch.Tensor]]]:

        # print(
        #     f">>> ENTER KIVI FORWARD | layer {self.layer_idx}"
        #     f" | q_len={hidden_states.shape[1]}"
        # )

        # print(
        #     ">>> RAW ARGUMENTS:",
        #     f"position_embeddings type={type(position_embeddings)}",
        #     f"attention_mask type={type(attention_mask)}, shape={getattr(attention_mask, 'shape', None)}",
        #     f"past_key_values is None={past_key_values is None}",
        #     f"cache_position={cache_position}",
        #     f"kwargs keys={list(kwargs.keys())}",
        # )
        is_prefill = cache_position is not None and cache_position.numel() > 1
        is_decode  = cache_position is not None and cache_position.numel() == 1


        # print(
        #     f">>> layer {self.layer_idx}"
        #     f" | is_prefill={is_prefill}"
        #     f" | is_decode={is_decode}"
        #     f" | cache_position shape={getattr(cache_position, 'shape', None)}"
        # )
        # Get the real HF attention mask (not the broken one)
        hf_mask = kwargs.get("attention_mask", None)

        bsz, q_len, _ = hidden_states.size()

        # Compute Q, K, V
        query_states = self.q_proj(hidden_states)
        key_states = self.k_proj(hidden_states)
        value_states = self.v_proj(hidden_states)

        # Reshape to [batch, seq_len, num_heads, head_dim]
        query_states = query_states.view(bsz, q_len, self.num_heads, self.head_dim).transpose(1, 2)
        key_states = key_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)
        value_states = value_states.view(bsz, q_len, self.num_key_value_heads, self.head_dim).transpose(1, 2)

        # Apply rotary embeddings
        # Apply rotary embeddings (correct for your HF version)

        # kv_seq_len = key_states.shape[-2]
        cos, sin = position_embeddings
        query_states, key_states = apply_rotary_pos_emb(
            query_states, key_states, cos, sin, None
        )


        # ========== KIVI CACHE LOGIC ==========
        if past_key_values is not None and self.use_kivi:

            if bsz != 1:
                raise NotImplementedError("KIVI currently only supports batch_size=1")

            # Initialize cache if needed
            self._init_kivi_cache()

            # Determine if this is prefill or decode

            if is_prefill:
                # ========== PREFILL PHASE ==========
                # Store KV cache for each head
                for kv_head_idx in range(self.num_key_value_heads):
                    # Extract [seq_len, head_dim]
                    head_keys = key_states[0, kv_head_idx, :, :].contiguous()
                    head_values = value_states[0, kv_head_idx, :, :].contiguous()

                    # Initialize KIVI cache
                    self.kivi_cache[kv_head_idx].prefill(head_keys, head_values)

                # print(f"\n[PREFILL] Layer {self.layer_idx}")
                # for i, cache in enumerate(self.kivi_cache):
                    # print(f"  Head {i} Key lengths:", cache.key_lengths())
                    # print(f"           Value lengths:", cache.value_lengths())

                # For prefill, use standard attention (pass exact KV for accuracy)
                # Repeat KV heads for GQA
                key_states = repeat_kv(key_states, self.num_key_value_groups)
                value_states = repeat_kv(value_states, self.num_key_value_groups)

                attn_output = self._standard_attention(
                    query_states, key_states, value_states, hf_mask
                )

            else:
                # ========== DECODE PHASE ==========
                attn_outputs = []

                for q_head_idx in range(self.num_heads):
                    # Map query head to KV head (for GQA)
                    kv_head_idx = q_head_idx // self.num_key_value_groups

                    # Extract query: [1, head_dim]
                    head_query = query_states[0, q_head_idx, :, :].contiguous()

                    # Extract new key/value: [1, head_dim]
                    head_key = key_states[0, kv_head_idx, :, :].contiguous()
                    head_value = value_states[0, kv_head_idx, :, :].contiguous()

                    # Update KIVI cache
                    self.kivi_cache[kv_head_idx].update(head_key, head_value)

                    # print(f"[DECODE] Layer {self.layer_idx} Q-head {q_head_idx} → KV-head {kv_head_idx}")
                    # print("  Key lengths:", self.kivi_cache[kv_head_idx].key_lengths())
                    # print("  Value lengths:", self.kivi_cache[kv_head_idx].value_lengths())

                    # Compute attention using KIVI cache
                    head_output = self.kivi_cache[kv_head_idx].compute_attention(
                        head_query,
                        use_split=False,
                        attention_mask=hf_mask  # passed down
                    )

                    attn_outputs.append(head_output)

                # Stack: [num_heads, 1, head_dim] -> [1, num_heads, 1, head_dim]
                attn_output = torch.stack(attn_outputs, dim=0).unsqueeze(0)

        else:
            # ========== STANDARD ATTENTION (NO KIVI) ==========
            key_states = repeat_kv(key_states, self.num_key_value_groups)
            value_states = repeat_kv(value_states, self.num_key_value_groups)

            attn_output = self._standard_attention(
                query_states, key_states, value_states, hf_mask
            )

        # Reshape back: [bsz, num_heads, seq_len, head_dim] -> [bsz, seq_len, hidden_size]
        attn_output = attn_output.transpose(1, 2).contiguous()
        attn_output = attn_output.reshape(bsz, q_len, self.hidden_size)

        # Output projection
        attn_output = self.o_proj(attn_output)

        return attn_output, None

    def _standard_attention(
        self,
        query_states: torch.Tensor,
        key_states: torch.Tensor,
        value_states: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Standard scaled dot-product attention with:
        - causal masking
        - (optional) external attention mask
        """
        # Compute raw attention scores
        attn_weights = torch.matmul(query_states, key_states.transpose(2, 3))
        attn_weights = attn_weights / math.sqrt(self.head_dim)

        # -------------------------------
        # 1. ADD CAUSAL MASK (IMPORTANT)
        # -------------------------------
        bsz, num_heads, q_len, _ = attn_weights.shape
        kv_len = key_states.shape[-2]

        # [q_len, kv_len], filled with -inf above diagonal
        causal_mask = torch.full(
            (q_len, kv_len),
            float("-inf"),
            device=attn_weights.device,
            dtype=attn_weights.dtype,
        )
        causal_mask = torch.triu(causal_mask, diagonal=1)  # leave lower triangle = 0

        # broadcast -> [1, 1, q_len, kv_len]
        attn_weights = attn_weights + causal_mask

        # -------------------------------
        # 2. ADD HF attention_mask if present
        # -------------------------------
        if attention_mask is not None:
            # mask shape normally [bsz, 1, q_len, kv_len]
            attn_weights = attn_weights + attention_mask

        # -------------------------------
        # 3. Softmax
        # -------------------------------
        attn_weights = nn.functional.softmax(
            attn_weights, dim=-1, dtype=torch.float32
        ).to(query_states.dtype)

        attn_weights = nn.functional.dropout(
            attn_weights, p=self.attention_dropout, training=self.training
        )

        # -------------------------------
        # 4. Multiply by values
        # -------------------------------
        attn_output = torch.matmul(attn_weights, value_states)
        return attn_output



# ========== HELPER FUNCTIONS ==========

def repeat_kv(hidden_states: torch.Tensor, n_rep: int) -> torch.Tensor:
    """
    Repeat key/value heads for Grouped Query Attention (GQA)
    hidden_states: [batch, num_key_value_heads, slen, head_dim]
    returns: [batch, num_attention_heads, slen, head_dim]
    """
    batch, num_key_value_heads, slen, head_dim = hidden_states.shape
    if n_rep == 1:
        return hidden_states
    hidden_states = hidden_states[:, :, None, :, :].expand(batch, num_key_value_heads, n_rep, slen, head_dim)
    return hidden_states.reshape(batch, num_key_value_heads * n_rep, slen, head_dim)


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)


def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None):
    """Apply rotary position embeddings to query and key tensors."""
    # cos, sin: [batch_size, seq_len, head_dim]
    cos = cos.unsqueeze(1)  # [bs, 1, seq_len, head_dim]
    sin = sin.unsqueeze(1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

def reset_all_kivi_caches(model):
    """Reset KIVI caches for all attention layers in a LLaMA model."""
    for layer in model.model.layers:
        attn = layer.self_attn
        if hasattr(attn, "reset_kivi_cache"):
            attn.reset_kivi_cache()


### Replacing LLaMA Attention with KIVI


In [ ]:
from transformers import LlamaForCausalLM, AutoTokenizer

def replace_llama_attention_with_kivi(model):
    """
    Replace all LlamaAttention layers with KIVI-enabled versions

    Args:
        model: LlamaForCausalLM model

    Returns:
        Modified model with KIVI attention
    """
    print("Replacing attention layers with KIVI...")

    for layer_idx, layer in enumerate(model.model.layers):
        # Get original attention
        original_attn = layer.self_attn

        # Create KIVI attention
        kivi_attn = LlamaAttentionWithKIVI(model.config, layer_idx=layer_idx)

        # Copy weights from original attention
        kivi_attn.q_proj.weight.data = original_attn.q_proj.weight.data.clone()
        kivi_attn.k_proj.weight.data = original_attn.k_proj.weight.data.clone()
        kivi_attn.v_proj.weight.data = original_attn.v_proj.weight.data.clone()
        kivi_attn.o_proj.weight.data = original_attn.o_proj.weight.data.clone()

        if hasattr(original_attn.q_proj, 'bias') and original_attn.q_proj.bias is not None:
            kivi_attn.q_proj.bias.data = original_attn.q_proj.bias.data.clone()
            kivi_attn.k_proj.bias.data = original_attn.k_proj.bias.data.clone()
            kivi_attn.v_proj.bias.data = original_attn.v_proj.bias.data.clone()
            kivi_attn.o_proj.bias.data = original_attn.o_proj.bias.data.clone()

        # Replace
        layer.self_attn = kivi_attn

        print(f"  Layer {layer_idx}: ✓")

    print("✅ All attention layers replaced with KIVI!")
    return model

## 4. Load Model and Run Inference


In [ ]:
import torch
from transformers import AutoTokenizer, LlamaForCausalLM
from huggingface_hub import login

# Step 1: Login with your token
login(token="") # put your token here

# Step 2: Load Llama-2-7b
model_name = "meta-llama/Llama-2-7b-chat-hf"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = LlamaForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)

print(f"✅ Model loaded successfully!")
print(f"   Parameters: {model.num_parameters() / 1e9:.2f}B")
print(f"   Layers: {len(model.model.layers)}")




### General Utility Functions


In [ ]:
def measure_baseline_memory(
    model,
    tokenizer,
    prompt,
    max_new_tokens=40,
):
    # ---- GPU peak ----
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    text, past_kv = generate_baseline_with_output_and_kv(
        model, tokenizer, prompt, max_new_tokens
    )

    gpu_peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    # ---- Theoretical KV ----
    kv_stats = get_baseline_kv_memory_stats(past_kv)

    return {
        "output": text,
        "gpu_peak_mb": gpu_peak_mb,
        "kv_theoretical_mb": kv_stats["total_mb"],
        "kv_stats": kv_stats,
    }

def measure_kivi_memory(
    model,
    tokenizer,
    prompt,
    max_new_tokens=40,
):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        _ = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    gpu_peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    # ---- KIVI memory ----
    kivi_theoretical = get_kivi_memory_stats(model)
    kivi_gpu = get_kivi_memory_stats_gpu(model)

    return {
        "gpu_peak_mb": gpu_peak_mb,
        "kv_theoretical_mb": kivi_theoretical["total_mb"],
        "kv_gpu_mb": kivi_gpu["total_mb"],
        "kv_theoretical_stats": kivi_theoretical,
        "kv_gpu_stats": kivi_gpu,
    }
def summarize_mb(values):
    return {
        "mean": float(np.mean(values)),
        "std": float(np.std(values)),
    }

def aggregate_memory(per_sample):
    baseline_gpu = []
    baseline_kv = []
    kivi_gpu = []
    kivi_kv_theory = []
    kivi_kv_gpu = []

    for ex in per_sample:
        m = ex["memory"]
        baseline_gpu.append(m["baseline"]["gpu_peak_mb"])
        baseline_kv.append(m["baseline"]["kv_theoretical_mb"])
        kivi_gpu.append(m["kivi"]["gpu_peak_mb"])
        kivi_kv_theory.append(m["kivi"]["kv_theoretical_mb"])
        kivi_kv_gpu.append(m["kivi"]["kv_gpu_mb"])

    return {
        "baseline_gpu_peak": summarize_mb(baseline_gpu),
        "baseline_kv_theoretical": summarize_mb(baseline_kv),
        "kivi_gpu_peak": summarize_mb(kivi_gpu),
        "kivi_kv_theoretical": summarize_mb(kivi_kv_theory),
        "kivi_kv_gpu": summarize_mb(kivi_kv_gpu),
    }
def print_memory_table(mem):
    print("\n📊 KV MEMORY COMPARISON")
    print(f"{'Metric':<35} {'Mean (MB)':>12} {'Std':>10}")
    print("-" * 60)

    def row(name, x):
        print(f"{name:<35} {x['mean']:>12.2f} {x['std']:>10.2f}")

    row("Baseline GPU peak", mem["baseline_gpu_peak"])
    row("Baseline KV (theoretical)", mem["baseline_kv_theoretical"])
    row("KIVI GPU peak", mem["kivi_gpu_peak"])
    row("KIVI KV (theoretical, bit-packed)", mem["kivi_kv_theoretical"])
    row("KIVI KV (actual GPU)", mem["kivi_kv_gpu"])

def theoretical_baseline_kv_mb(
    num_prompt_tokens: int,
    num_generated_tokens: int,
    num_layers: int,
    num_heads: int,
    head_dim: int,
    dtype_bytes: int = 2,  # fp16
):
    """
    Theoretical FP16 KV cache memory for baseline attention.

    KV shape per layer:
      [num_heads, total_tokens, head_dim] for K
      [num_heads, total_tokens, head_dim] for V
    """
    total_tokens = num_prompt_tokens + num_generated_tokens

    total_bytes = (
        total_tokens
        * num_layers
        * num_heads
        * head_dim
        * dtype_bytes
        * 2  # K + V
    )

    return {
        "total_tokens": total_tokens,
        "total_bytes": total_bytes,
        "total_mb": total_bytes / (1024 ** 2),
        "total_gb": total_bytes / (1024 ** 3),
    }

def get_model_kv_config(model):
    cfg = model.config
    return {
        "num_layers": cfg.num_hidden_layers,
        "num_heads": cfg.num_attention_heads,
        "head_dim": cfg.hidden_size // cfg.num_attention_heads,
    }

def get_kivi_memory_stats(model, verbose=False):
    """
    Compute total memory usage across ALL KIVI caches in the model.
    """
    totals = {
        "total_bytes": 0,
        "key_residual_bytes": 0,
        "value_residual_bytes": 0,
        "key_quant_bytes": 0,
        "value_quant_bytes": 0,
        "key_metadata_bytes": 0,
        "value_metadata_bytes": 0,
        "layers_with_kivi": 0,
        "heads_per_layer": [],
    }

    for layer_idx, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        if not hasattr(attn, "kivi_cache") or attn.kivi_cache is None:
            continue  # Layer does not use KIVI

        totals["layers_with_kivi"] += 1
        totals["heads_per_layer"].append(len(attn.kivi_cache))

        for head_idx, cache in enumerate(attn.kivi_cache):
            stats = cache.get_memory_stats()

            totals["total_bytes"] += stats["total_bytes"]
            totals["key_residual_bytes"] += stats["key_residual_bytes"]
            totals["value_residual_bytes"] += stats["value_residual_bytes"]
            totals["key_quant_bytes"] += stats["key_quant_bytes"]
            totals["value_quant_bytes"] += stats["value_quant_bytes"]
            totals["key_metadata_bytes"] += stats["key_metadata_bytes"]
            totals["value_metadata_bytes"] += stats["value_metadata_bytes"]

            if verbose:
                print(f"[Layer {layer_idx}][Head {head_idx}] {stats}")

    totals["total_mb"] = totals["total_bytes"] / (1024 ** 2)
    totals["total_gb"] = totals["total_bytes"] / (1024 ** 3)

    return totals

In [ ]:
def get_kivi_memory_stats(model, verbose=False):
    """
    Compute total memory usage across ALL KIVI caches in the model.
    """
    totals = {
        "total_bytes": 0,
        "key_residual_bytes": 0,
        "value_residual_bytes": 0,
        "key_quant_bytes": 0,
        "value_quant_bytes": 0,
        "key_metadata_bytes": 0,
        "value_metadata_bytes": 0,
        "layers_with_kivi": 0,
        "heads_per_layer": [],
    }

    for layer_idx, layer in enumerate(model.model.layers):
        attn = layer.self_attn
        if not hasattr(attn, "kivi_cache") or attn.kivi_cache is None:
            continue  # Layer does not use KIVI

        totals["layers_with_kivi"] += 1
        totals["heads_per_layer"].append(len(attn.kivi_cache))

        for head_idx, cache in enumerate(attn.kivi_cache):
            stats = cache.get_memory_stats()

            totals["total_bytes"] += stats["total_bytes"]
            totals["key_residual_bytes"] += stats["key_residual_bytes"]
            totals["value_residual_bytes"] += stats["value_residual_bytes"]
            totals["key_quant_bytes"] += stats["key_quant_bytes"]
            totals["value_quant_bytes"] += stats["value_quant_bytes"]
            totals["key_metadata_bytes"] += stats["key_metadata_bytes"]
            totals["value_metadata_bytes"] += stats["value_metadata_bytes"]

            if verbose:
                print(f"[Layer {layer_idx}][Head {head_idx}] {stats}")

    totals["total_mb"] = totals["total_bytes"] / (1024 ** 2)
    totals["total_gb"] = totals["total_bytes"] / (1024 ** 3)

    return totals

def print_kivi_memory_stats(stats):
    print("=" * 70)
    print("📊  KIVI Cache Memory Statistics")
    print("=" * 70)
    print(f"Total memory:          {stats['total_mb']:.2f} MB ({stats['total_gb']:.4f} GB)")
    print(f"Layers with KIVI:      {stats['layers_with_kivi']}")
    print(f"Heads per layer:       {stats['heads_per_layer']}")
    print("-" * 70)
    print(f"Key residual:          {stats['key_residual_bytes'] / (1024**2):.2f} MB")
    print(f"Value residual:        {stats['value_residual_bytes'] / (1024**2):.2f} MB")
    print(f"Key quantized:         {stats['key_quant_bytes'] / (1024**2):.2f} MB")
    print(f"Value quantized:       {stats['value_quant_bytes'] / (1024**2):.2f} MB")
    print(f"Key metadata:          {stats['key_metadata_bytes'] / (1024**2):.2f} MB")
    print(f"Value metadata:        {stats['value_metadata_bytes'] / (1024**2):.2f} MB")
    print("=" * 70)

def get_baseline_kv_memory_stats(past_key_values, dtype_bytes=2):
    """
    Compute memory usage of baseline FP16 KV cache.

    Args:
        past_key_values: tuple from HF generation
        dtype_bytes: bytes per element (fp16 = 2)

    Returns:
        dict with memory stats
    """
    stats = {
        "key_bytes": 0,
        "value_bytes": 0,
        "total_bytes": 0,
    }

    for layer_idx, (k, v) in enumerate(past_key_values):
        # k, v shape: [batch, num_heads, seq_len, head_dim]
        stats["key_bytes"] += k.numel() * dtype_bytes
        stats["value_bytes"] += v.numel() * dtype_bytes

    stats["total_bytes"] = stats["key_bytes"] + stats["value_bytes"]
    stats["total_mb"] = stats["total_bytes"] / (1024 ** 2)
    stats["total_gb"] = stats["total_bytes"] / (1024 ** 3)

    return stats

def generate_with_kv_capture(model, tokenizer, prompt, max_new_tokens=40):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    past_key_values = None
    input_ids = inputs["input_ids"]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(
                input_ids=input_ids,
                past_key_values=past_key_values,
                use_cache=True,
            )
            logits = outputs.logits
            past_key_values = outputs.past_key_values

            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            input_ids = next_token

    return past_key_values

def generate_baseline_with_output_and_kv(
    model,
    tokenizer,
    prompt,
    max_new_tokens=40,
):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_ids = inputs["input_ids"]
    prompt_len = input_ids.shape[-1]   # 🔹 ADD THIS

    past_key_values = None
    generated_ids = input_ids.clone()

    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(
                input_ids=input_ids,
                past_key_values=past_key_values,
                use_cache=True,
            )

            logits = outputs.logits
            past_key_values = outputs.past_key_values

            next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
            generated_ids = torch.cat([generated_ids, next_token], dim=-1)
            input_ids = next_token

    # 🔹 SLICE OFF PROMPT HERE
    continuation_ids = generated_ids[:, prompt_len:]
    text = tokenizer.decode(continuation_ids[0], skip_special_tokens=True)

    return text, past_key_values




### Short Prompt Test


In [ ]:
import torch
from transformers import AutoTokenizer, LlamaForCausalLM

# === Import your modules ===
# (assuming they are in the same notebook or directory)
# from kivi_attention import replace_llama_attention_with_kivi
# from kivi_memory_stats import get_kivi_memory_stats, print_kivi_memory_stats

# ----------------------------------------------------------
# Simple generation helper
# ----------------------------------------------------------
def generate_text(model, tokenizer, prompt, label, max_new_tokens=40):
    print("\n==================================================")
    print(f"[{label}] Prompt: {prompt}")
    print("==================================================")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 🔹 Only decode NEW tokens
    generated_ids = outputs[0][prompt_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[{label}] Generated continuation:\n{text}\n")
    return text


# ----------------------------------------------------------
# Main benchmark
# ----------------------------------------------------------
def main():
    model_name = "meta-llama/Llama-2-7b-chat-hf"
    print(f"Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # ======== BASELINE MODEL ========
    print("\n====================")
    print(" Loading BASELINE")
    print("====================")
    baseline_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    # ======== KIVI MODEL ========
    print("\n====================")
    print(" Loading KIVI MODEL")
    print("====================")
    kivi_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    kivi_model = replace_llama_attention_with_kivi(kivi_model)
    print("✔ KIVI attention injected!")

    # ======== Test Prompts ========
    prompts = [
        "The capital of France is",
        "In machine learning, attention mechanisms",
        "The Eiffel Tower is located in",
    ]

    # ======== Run Tests ========
    results = {}

    for prompt in prompts:
        prompt_len = inputs["input_ids"].shape[-1]
        generated_ids = generated_ids[:, prompt_len:]
        text = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
        base_out = generate_text(baseline_model, tokenizer, prompt, label="BASELINE")
        kivi_out = generate_text(kivi_model, tokenizer, prompt, label="KIVI")

        results[prompt] = {
            "baseline": base_out,
            "kivi": kivi_out
        }

    # ======== Print side-by-side summary ========
    print("\n\n====================")
    print(" SUMMARY COMPARISON")
    print("====================")
    for prompt, outs in results.items():
        print("\n----------------------------------------")
        print(f"Prompt: {prompt}")
        print("----------------------------------------")
        print("BASELINE:")
        print(outs["baseline"])
        print("\nKIVI:")
        print(outs["kivi"])
        print("----------------------------------------")

    # ======== Memory usage (KIVI only) ========
    print("\n\n================================================")
    print("       📊 KIVI Cache Memory Statistics")
    print("================================================")
    stats = get_kivi_memory_stats(kivi_model)
    print_kivi_memory_stats(stats)


if __name__ == "__main__":
    main()


### Long Prompt Test


In [ ]:
import torch
from transformers import AutoTokenizer, LlamaForCausalLM

# === Import your modules ===
# (assuming they are in the same notebook or directory)
# from kivi_attention import replace_llama_attention_with_kivi
# from kivi_memory_stats import get_kivi_memory_stats, print_kivi_memory_stats

# ----------------------------------------------------------
# Simple generation helper
# ----------------------------------------------------------
def generate_text(model, tokenizer, prompt, label, max_new_tokens=40):
    print("\n==================================================")
    print(f"[{label}] Prompt: {prompt}")
    print("==================================================")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 🔹 Only decode NEW tokens
    generated_ids = outputs[0][prompt_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[{label}] Generated continuation:\n{text}\n")
    return text


# ----------------------------------------------------------
# Main benchmark
# ----------------------------------------------------------
def main():
    model_name = "meta-llama/Llama-2-7b-chat-hf"
    print(f"Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # ======== BASELINE MODEL ========
    print("\n====================")
    print(" Loading BASELINE")
    print("====================")
    baseline_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    # ======== KIVI MODEL ========
    print("\n====================")
    print(" Loading KIVI MODEL")
    print("====================")
    kivi_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    kivi_model = replace_llama_attention_with_kivi(kivi_model)
    print("✔ KIVI attention injected!")

    prompt = "Antibiotics are a type of medication used to treat bacterial infections. They work by either killing the bacteria or preventing them from reproducing, allowing the body’s immune system to fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or liquid solutions, or sometimes administered intravenously. They are not effective against viral infections, and using them inappropriately can lead to antibiotic resistance. Explain the above in one sentence:"
    max_new_tokens = 40


    # ================= BASELINE =================
    baseline_text, baseline_pkv = generate_baseline_with_output_and_kv(
        baseline_model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
    )

    print("\n==================================================")
    print("[BASELINE] Prompt:")
    print(prompt)
    print("==================================================")
    print("[BASELINE] Output:")
    print(baseline_text)

    baseline_stats = get_baseline_kv_memory_stats(baseline_pkv)

    print("\n==============================")
    print("📊 BASELINE KV CACHE MEMORY")
    print("==============================")
    print(f"Baseline KV total: {baseline_stats['total_mb']:.2f} MB")
    print(f"  Keys:   {baseline_stats['key_bytes'] / (1024**2):.2f} MB")
    print(f"  Values: {baseline_stats['value_bytes'] / (1024**2):.2f} MB")

    # ================= KIVI =================
    _ = generate_text(kivi_model, tokenizer, prompt, label="KIVI", max_new_tokens=max_new_tokens)

    kivi_stats = get_kivi_memory_stats(kivi_model)

    print("\n==============================")
    print("📊 KIVI KV CACHE MEMORY")
    print("==============================")
    print_kivi_memory_stats(kivi_stats)


    # ================= REDUCTION =================
    print("\n==============================")
    print("📉 MEMORY REDUCTION")
    print("==============================")

    reduction = baseline_stats["total_bytes"] / kivi_stats["total_bytes"]
    print(f"Memory reduction factor: {reduction:.2f}×")



if __name__ == "__main__":
    main()


In [ ]:
import torch
from transformers import AutoTokenizer, LlamaForCausalLM

# === Import your modules ===
# (assuming they are in the same notebook or directory)
# from kivi_attention import replace_llama_attention_with_kivi
# from kivi_memory_stats import get_kivi_memory_stats, print_kivi_memory_stats

# ----------------------------------------------------------
# Simple generation helper
# ----------------------------------------------------------
def generate_text(model, tokenizer, prompt, label, max_new_tokens=40):
    print("\n==================================================")
    print(f"[{label}] Prompt: {prompt}")
    print("==================================================")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
        )

    # 🔹 Only decode NEW tokens
    generated_ids = outputs[0][prompt_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[{label}] Generated continuation:\n{text}\n")
    return text


# ----------------------------------------------------------
# Main benchmark
# ----------------------------------------------------------
def main():
    model_name = "meta-llama/Llama-2-7b-chat-hf"
    print(f"Loading model: {model_name}")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # ======== BASELINE MODEL ========
    print("\n====================")
    print(" Loading BASELINE")
    print("====================")
    baseline_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    # ======== KIVI MODEL ========
    print("\n====================")
    print(" Loading KIVI MODEL")
    print("====================")
    kivi_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    kivi_model = replace_llama_attention_with_kivi(kivi_model)
    print("✔ KIVI attention injected!")

    prompt = "Antibiotics are a type of medication used to treat bacterial infections. They work by either killing the bacteria or preventing them from reproducing, allowing the body’s immune system to fight off the infection. Antibiotics are usually taken orally in the form of pills, capsules, or liquid solutions, or sometimes administered intravenously. They are not effective against viral infections, and using them inappropriately can lead to antibiotic resistance. Explain the above in one sentence:"
    max_new_tokens = 40


    # ================= BASELINE =================
    baseline_text, baseline_pkv = generate_baseline_with_output_and_kv(
        baseline_model,
        tokenizer,
        prompt,
        max_new_tokens=max_new_tokens,
    )

    print("\n==================================================")
    print("[BASELINE] Prompt:")
    print(prompt)
    print("==================================================")
    print("[BASELINE] Output:")
    print(baseline_text)

    baseline_stats = get_baseline_kv_memory_stats(baseline_pkv)

    print("\n==============================")
    print("📊 BASELINE KV CACHE MEMORY")
    print("==============================")
    print(f"Baseline KV total: {baseline_stats['total_mb']:.2f} MB")
    print(f"  Keys:   {baseline_stats['key_bytes'] / (1024**2):.2f} MB")
    print(f"  Values: {baseline_stats['value_bytes'] / (1024**2):.2f} MB")

    # ================= KIVI =================
    kivi_text = generate_text(
        kivi_model,
        tokenizer,
        prompt,
        label="KIVI",
        max_new_tokens=max_new_tokens,
    )

    kivi_stats = get_kivi_memory_stats(kivi_model)

    print("\n==============================")
    print("📊 KIVI KV CACHE MEMORY")
    print("==============================")
    print_kivi_memory_stats(kivi_stats)


    # ================= QUALITY METRICS =================
    print("\n==============================")
    print("📐 QUALITY METRICS (Baseline vs KIVI)")
    print("==============================")

    # Token-level agreement
    token_stats = token_level_match_rate(
        baseline_text,
        kivi_text,
        tokenizer,
    )

    print(f"Token match rate: {token_stats['token_match_rate']:.4f}")
    print(f"Matched tokens:   {token_stats['matched_tokens']} / {token_stats['compared_tokens']}")
    print(f"Baseline length:  {token_stats['baseline_len']}")
    print(f"KIVI length:      {token_stats['kivi_len']}")

    # ROUGE-L
    rougeL = rouge_l_score(baseline_text, kivi_text)
    print(f"ROUGE-L F1:       {rougeL:.4f}")

    # 🔹 BERTScore (semantic similarity)
    bert_f1 = bert_score_f1(
        reference_text=baseline_text,
        prediction_text=kivi_text,
    )
    print(f"BERTScore F1:     {bert_f1:.4f}")



if __name__ == "__main__":
    main()


## 5. CNN/DailyMail Summarization Benchmark


### Helper Functions


In [ ]:
from datasets import load_dataset

def load_cnn_dm(split="validation", num_samples=10):
    dataset = load_dataset("cnn_dailymail", "3.0.0", split=split)
    return dataset.select(range(num_samples))

def build_summarization_prompt(tokenizer, article):
    messages = [
        {"role": "system", "content": "You are a helpful assistant that writes concise summaries."},
        {"role": "user", "content": f"Summarize the following news article in 3 sentences.\n\n{article}\n\nSUMMARY:"}
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


In [ ]:
pip install rouge-score

In [ ]:
pip install bert-score

In [ ]:
def token_level_match_rate(
    baseline_text,
    kivi_text,
    tokenizer,
):
    # Tokenize WITHOUT adding BOS/EOS again
    base_tokens = tokenizer.encode(baseline_text, add_special_tokens=False)
    kivi_tokens = tokenizer.encode(kivi_text, add_special_tokens=False)

    min_len = min(len(base_tokens), len(kivi_tokens))

    matches = sum(
        base_tokens[i] == kivi_tokens[i]
        for i in range(min_len)
    )

    return {
        "token_match_rate": matches / min_len if min_len > 0 else 0.0,
        "matched_tokens": matches,
        "compared_tokens": min_len,
        "baseline_len": len(base_tokens),
        "kivi_len": len(kivi_tokens),
    }

from rouge_score import rouge_scorer

def rouge_l_score(reference, prediction):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = scorer.score(reference, prediction)
    return scores["rougeL"].fmeasure

from bert_score import score as bertscore

def bert_score_f1(
    reference_text,
    prediction_text,
    # model_type="microsoft/deberta-xlarge-mnli",
    model_type="roberta-large",
    device="cpu",
):
    """
    Compute BERTScore F1 between reference and prediction.

    Returns:
        float: BERTScore F1
    """
    if device is None:
        device = "cpu"

    P, R, F1 = bertscore(
        [prediction_text],
        [reference_text],
        lang="en",
        model_type=model_type,
        device=device,
        verbose=False,
    )

    return F1.mean().item()

import re
from transformers import pipeline

def split_sentences(text: str):
    # Simple, robust sentence splitter (no nltk dependency)
    text = text.strip()
    if not text:
        return []
    # Split on ., !, ? followed by whitespace/newline
    sents = re.split(r'(?<=[.!?])\s+', text)
    # Clean up
    sents = [s.strip() for s in sents if s.strip()]
    return sents

def sentence_compliance_error(generated: str, requested: int = 3) -> int:
    n_gen = len(split_sentences(generated))
    return abs(n_gen - requested)

def is_sentence_compliant(generated: str, requested: int = 3) -> int:
    return int(sentence_compliance_error(generated, requested) == 0)

def sentence_compliance_score(
    generated: str,
    requested: int = 3,
    max_penalty: int = 3,
) -> float:
    n_gen = len(split_sentences(generated))
    diff = abs(n_gen - requested)

    score = 1.0 - diff / max_penalty
    return max(0.0, score)


def normalize_for_match(s: str) -> str:
    return re.sub(r'\s+', ' ', s.lower()).strip()

def extract_summary_entities(summary: str):
    """
    Returns a set of entity-like strings from summary:
    - numbers / ages / years (e.g., 27, 12, 2019, 26-70)
    - capitalized spans (e.g., "Zully Broussard", "California Pacific Medical Center", "Louisiana")
    """
    ents = set()

    # Numbers / numeric ranges / ages
    for m in re.findall(r'\b\d+(?:[\-–]\d+)?\b', summary):
        ents.add(m)

    # Capitalized spans (simple NER-ish heuristic)
    # Matches: "Zully Broussard", "California Pacific Medical Center", etc.
    cap_spans = re.findall(r'\b(?:[A-Z][a-z]+(?:\s+[A-Z][a-z]+){0,5})\b', summary)
    for s in cap_spans:
        # filter out sentence-start common words
        if s in {"The", "A", "An", "This", "That", "In", "On", "But", "And", "Some"}:
            continue
        ents.add(s.strip())

    return ents

def entity_hallucination_rate(article: str, summary: str):
    """
    Fraction of extracted entities in summary that do NOT appear in article text.
    Returns:
      rate, hallucinated_list, matched_list
    """
    art = normalize_for_match(article)
    ents = extract_summary_entities(summary)
    if len(ents) == 0:
        return 0.0, [], []

    hallucinated = []
    matched = []
    for e in ents:
        if normalize_for_match(e) in art:
            matched.append(e)
        else:
            hallucinated.append(e)

    rate = len(hallucinated) / max(1, len(ents))
    return rate, hallucinated, matched

def chunk_text_by_words(text: str, chunk_words: int = 220, overlap: int = 40):
    words = text.split()
    if not words:
        return []
    chunks = []
    i = 0
    while i < len(words):
        chunk = words[i:i+chunk_words]
        chunks.append(" ".join(chunk))
        i += max(1, chunk_words - overlap)
    return chunks

# def nli_entailment_best_over_chunks(nli_pipe, article: str, summary: str,
#                                    chunk_words: int = 220, overlap: int = 40):
#     """
#     Returns:
#       entail_best, contradiction_best, neutral_best
#     Using MNLI labels: entailment / contradiction / neutral.
#     """
#     chunks = chunk_text_by_words(article, chunk_words=chunk_words, overlap=overlap)
#     if not chunks:
#         return 0.0, 0.0, 0.0

#     best_ent = 0.0
#     best_con = 0.0
#     best_neu = 0.0

#     for ch in chunks:
#         out = nli_pipe({"text": ch, "text_pair": summary}, truncation=True)
#         # out is list of dicts like [{'label': 'ENTAILMENT', 'score': ...}, ...]
#         scores = {d["label"].lower(): float(d["score"]) for d in out}
#         ent = scores.get("entailment", 0.0)
#         con = scores.get("contradiction", 0.0)
#         neu = scores.get("neutral", 0.0)

#         best_ent = max(best_ent, ent)
#         best_con = max(best_con, con)
#         best_neu = max(best_neu, neu)

#     return best_ent, best_con, best_neu




### Single Example Demo


In [ ]:
import numpy as np


def generate_text(model, tokenizer, prompt, label, max_new_tokens=40):
    print("\n==================================================")
    print(f"[{label}] Prompt: {prompt}")
    print("==================================================")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # 🔹 Only decode NEW tokens
    generated_ids = outputs[0][prompt_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    print(f"[{label}] Generated continuation:\n{text}\n")
    return text

def evaluate_on_cnn_dm(
    dataset,
    tokenizer,
    baseline_model,
    kivi_model,
    max_new_tokens=128,
):
    metrics = {
        "token_bk": [],
        "rouge_bk": [],
        "bert_bk": [],
        "rouge_bg": [],
        "bert_bg": [],
        "rouge_kg": [],
        "bert_kg": [],
    }

    for i, sample in enumerate(dataset):
        print(f"\n================ Example {i} =================")

        prompt = build_summarization_prompt(tokenizer,sample["article"])
        reference = sample["highlights"]

        baseline_text = generate_text(
            baseline_model, tokenizer, prompt, "BASELINE", max_new_tokens
        )
        kivi_text = generate_text(
            kivi_model, tokenizer, prompt, "KIVI", max_new_tokens
        )

        # Baseline ↔ KIVI
        token_stats = token_level_match_rate(baseline_text, kivi_text, tokenizer)
        rouge_bk = rouge_l_score(baseline_text, kivi_text)
        bert_bk  = bert_score_f1(baseline_text, kivi_text)

        # Baseline ↔ GT
        rouge_bg = rouge_l_score(reference, baseline_text)
        bert_bg  = bert_score_f1(reference, baseline_text)

        # KIVI ↔ GT
        rouge_kg = rouge_l_score(reference, kivi_text)
        bert_kg  = bert_score_f1(reference, kivi_text)

        metrics["token_bk"].append(token_stats["token_match_rate"])
        metrics["rouge_bk"].append(rouge_bk)
        metrics["bert_bk"].append(bert_bk)
        metrics["rouge_bg"].append(rouge_bg)
        metrics["bert_bg"].append(bert_bg)
        metrics["rouge_kg"].append(rouge_kg)
        metrics["bert_kg"].append(bert_kg)

    return {k: float(np.mean(v)) for k, v in metrics.items()}


In [ ]:
import torch
from transformers import AutoTokenizer, LlamaForCausalLM



def main():
    model_name = "meta-llama/Llama-2-7b-chat-hf"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # ======== BASELINE ========
    baseline_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    # ======== KIVI ========
    kivi_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    kivi_model = replace_llama_attention_with_kivi(kivi_model)

    # ---- One CNN article ----
    cnn_dm = load_cnn_dm(split="validation", num_samples=1)

    # ---- Quality evaluation ----
    results = evaluate_on_cnn_dm(
        dataset=cnn_dm,
        tokenizer=tokenizer,
        baseline_model=baseline_model,
        kivi_model=kivi_model,
        max_new_tokens=128,
    )

    print("\n📊 METRIC COMPARISON (CNN/DailyMail)")
    print(f"{'Metric':<20} {'Baseline vs GT':>18} {'KIVI vs GT':>18} {'Baseline vs KIVI':>22}")
    print("-" * 80)

    print(f"{'ROUGE-L F1':<20} "
        f"{results['rouge_bg']:>18.4f} "
        f"{results['rouge_kg']:>18.4f} "
        f"{results['rouge_bk']:>22.4f}")

    print(f"{'BERTScore F1':<20} "
        f"{results['bert_bg']:>18.4f} "
        f"{results['bert_kg']:>18.4f} "
        f"{results['bert_bk']:>22.4f}")

    print(f"{'Token match rate':<20} "
        f"{'—':>18} "
        f"{'—':>18} "
        f"{results['token_bk']:>22.4f}")

    # ---- Baseline KV memory (single example) ----
    sample = cnn_dm[0]
    prompt = build_summarization_prompt(tokenizer,sample["article"])

    reference = sample["highlights"]  # CNN/DM ground truth


    _, baseline_pkv = generate_baseline_with_output_and_kv(
        baseline_model,
        tokenizer,
        prompt,
        max_new_tokens=128,
    )

    baseline_stats = get_baseline_kv_memory_stats(baseline_pkv)

    print("\n==============================")
    print("📊 BASELINE KV CACHE MEMORY")
    print("==============================")
    print(f"Baseline KV total: {baseline_stats['total_mb']:.2f} MB")

    # ---- KIVI KV memory ----
    kivi_stats = get_kivi_memory_stats(kivi_model)

    print("\n==============================")
    print("📊 KIVI KV CACHE MEMORY")
    print("==============================")
    print_kivi_memory_stats(kivi_stats)





if __name__ == "__main__":
    main()


### Full Evaluation (100 Samples)


In [ ]:
import numpy as np

NUM_PRINT = 3

def generate_text(
    model,
    tokenizer,
    prompt,
    label,
    max_new_tokens=40,
    verbose=False,
):
    if verbose:
        print("\n==================================================")
        print(f"[{label}] Prompt: {prompt}")
        print("==================================================")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][prompt_len:]
    text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    if verbose:
        print(f"[{label}] Generated continuation:\n{text}\n")

    return text

def summarize(values):
    return {
        "mean": float(np.mean(values)),
        "std":  float(np.std(values)),
    }

from collections import defaultdict
import numpy as np
from transformers import pipeline

def evaluate_on_cnn_dm_from_texts(
    articles,
    references,
    prompts,
    baseline_outputs,
    kivi_outputs,
    tokenizer,
):
    # ---- NLI pipeline (CPU only) ----
    nli_pipe = pipeline(
        "text-classification",
        model="facebook/bart-large-mnli",
        tokenizer="facebook/bart-large-mnli",
        return_all_scores=True,
        device=-1,
    )

    metrics = defaultdict(list)
    per_sample = []

    for i in range(len(articles)):
        article = articles[i]
        reference = references[i]
        baseline_text = baseline_outputs[i]
        kivi_text = kivi_outputs[i]

        # --------------------------
        # Metrics
        # --------------------------
        token_stats = token_level_match_rate(baseline_text, kivi_text, tokenizer)

        rouge_bk = rouge_l_score(baseline_text, kivi_text)
        bert_bk  = bert_score_f1(baseline_text, kivi_text)

        rouge_bg = rouge_l_score(reference, baseline_text)
        bert_bg  = bert_score_f1(reference, baseline_text)

        rouge_kg = rouge_l_score(reference, kivi_text)
        bert_kg  = bert_score_f1(reference, kivi_text)

        sce_b = sentence_compliance_score(baseline_text, 3)
        sce_k = sentence_compliance_score(kivi_text, 3)

        # ent_b, con_b, _ = nli_entailment_best_over_chunks(nli_pipe, article, baseline_text)
        # ent_k, con_k, _ = nli_entailment_best_over_chunks(nli_pipe, article, kivi_text)

        hall_b, _, _ = entity_hallucination_rate(article, baseline_text)
        hall_k, _, _ = entity_hallucination_rate(article, kivi_text)

        # --------------------------
        # Save per-sample record ✅
        # --------------------------
        row = {
            "idx": i,

            # ---- TEXT ----
            "prompt": prompts[i],
            "article": article[:2000],
            "reference": reference,
            "baseline_output": baseline_text,
            "kivi_output": kivi_text,

            # ---- SCORES ----
            "scores": {
                "baseline_vs_gt": {
                    "rouge_l": rouge_bg,
                    "bert": bert_bg,
                    "sce": sce_b,
                    "scr": int(sce_b == 0),
                    # "nli_ent": ent_b,
                    # "nli_con": con_b,
                    "ent_hall_rate": hall_b,
                },
                "kivi_vs_gt": {
                    "rouge_l": rouge_kg,
                    "bert": bert_kg,
                    "sce": sce_k,
                    "scr": int(sce_k == 0),
                    # "nli_ent": ent_k,
                    # "nli_con": con_k,
                    "ent_hall_rate": hall_k,
                },
                "baseline_vs_kivi": {
                    "token_match_rate": token_stats["token_match_rate"],
                    "rouge_l": rouge_bk,
                    "bert": bert_bk,
                },
            },
        }

        per_sample.append(row)

        # --------------------------
        # Aggregate tracking
        # --------------------------
        metrics["rouge_bg"].append(rouge_bg)
        metrics["bert_bg"].append(bert_bg)
        metrics["rouge_kg"].append(rouge_kg)
        metrics["bert_kg"].append(bert_kg)
        metrics["rouge_bk"].append(rouge_bk)
        metrics["bert_bk"].append(bert_bk)
        metrics["token_bk"].append(token_stats["token_match_rate"])

        metrics["sce_baseline"].append(sce_b)
        metrics["sce_kivi"].append(sce_k)
        metrics["scr_baseline"].append(int(sce_b == 0))
        metrics["scr_kivi"].append(int(sce_k == 0))

        # metrics["nli_ent_baseline"].append(ent_b)
        # metrics["nli_ent_kivi"].append(ent_k)
        # metrics["nli_con_baseline"].append(con_b)
        # metrics["nli_con_kivi"].append(con_k)

        metrics["ent_hall_rate_baseline"].append(hall_b)
        metrics["ent_hall_rate_kivi"].append(hall_k)

    # --------------------------
    # Aggregate summaries
    # --------------------------
    def summarize(x):
        return {"mean": float(np.mean(x)), "std": float(np.std(x))}

    results = {
        "rouge_bg": summarize(metrics["rouge_bg"]),
        "bert_bg": summarize(metrics["bert_bg"]),
        "rouge_kg": summarize(metrics["rouge_kg"]),
        "bert_kg": summarize(metrics["bert_kg"]),
        "rouge_bk": summarize(metrics["rouge_bk"]),
        "bert_bk": summarize(metrics["bert_bk"]),
        "token_bk": summarize(metrics["token_bk"]),

        "sce_baseline": summarize(metrics["sce_baseline"]),
        "sce_kivi": summarize(metrics["sce_kivi"]),
        "scr_baseline": float(np.mean(metrics["scr_baseline"])),
        "scr_kivi": float(np.mean(metrics["scr_kivi"])),

        # "nli_ent_baseline": summarize(metrics["nli_ent_baseline"]),
        # "nli_ent_kivi": summarize(metrics["nli_ent_kivi"]),
        # "nli_con_baseline": summarize(metrics["nli_con_baseline"]),
        # "nli_con_kivi": summarize(metrics["nli_con_kivi"]),

        "ent_hall_rate_baseline": summarize(metrics["ent_hall_rate_baseline"]),
        "ent_hall_rate_kivi": summarize(metrics["ent_hall_rate_kivi"]),
    }

    return per_sample, results

import matplotlib.pyplot as plt

def plot_rouge(per_sample):
    x = [s["idx"] for s in per_sample]
    rouge_bg = [s["rouge_bg"] for s in per_sample]
    rouge_kg = [s["rouge_kg"] for s in per_sample]

    plt.figure()
    plt.plot(x, rouge_bg, marker="o", label="Baseline vs GT")
    plt.plot(x, rouge_kg, marker="o", label="KIVI vs GT")
    plt.xlabel("Sample index")
    plt.ylabel("ROUGE-L F1")
    plt.title("ROUGE-L per sample")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_bert(per_sample):
    x = [s["idx"] for s in per_sample]
    bert_bg = [s["bert_bg"] for s in per_sample]
    bert_kg = [s["bert_kg"] for s in per_sample]

    plt.figure()
    plt.plot(x, bert_bg, marker="o", label="Baseline vs GT")
    plt.plot(x, bert_kg, marker="o", label="KIVI vs GT")
    plt.xlabel("Sample index")
    plt.ylabel("BERTScore F1")
    plt.title("BERTScore per sample")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_token_match(per_sample):
    x = [s["idx"] for s in per_sample]
    token_bk = [s["token_bk"] for s in per_sample]

    plt.figure()
    plt.plot(x, token_bk, marker="o", color="purple")
    plt.xlabel("Sample index")
    plt.ylabel("Token match rate")
    plt.title("Baseline vs KIVI token match rate")
    plt.grid(True)
    plt.show()



In [ ]:
@torch.no_grad()
def generate_all_summaries(
    model,
    tokenizer,
    dataset,
    label,
    max_new_tokens,
    num_print=3,
):
    outputs = []
    prompts = []
    references = []
    kv_theoretical = []
    peak_gpu_mb_per_sample = []

    kv_cfg = get_model_kv_config(model)

    for i, sample in enumerate(dataset):
        article = sample["article"]
        prompt = build_summarization_prompt(tokenizer, article)

        # ---- TOKEN COUNTS (prompt) ----
        prompt_ids = tokenizer(
            prompt, return_tensors="pt", add_special_tokens=False
        )["input_ids"]
        num_prompt_tokens = prompt_ids.shape[-1]

        # 🔹 HARD RESET for THIS SAMPLE
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        # ---- GENERATION ----
        text = generate_text(
            model,
            tokenizer,
            prompt,
            label,
            max_new_tokens=max_new_tokens,
            verbose=(i < num_print),
        )

        # 🔹 Empirical peak for THIS sample ONLY
        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        peak_gpu_mb_per_sample.append(peak_mb)

        # ---- TOKEN COUNTS (generated) ----
        gen_ids = tokenizer(
            text, return_tensors="pt", add_special_tokens=False
        )["input_ids"]
        num_gen_tokens = gen_ids.shape[-1]

        # ---- THEORETICAL KV CACHE ----
        if label == "BASELINE":
            kv_stats = theoretical_baseline_kv_mb(
                num_prompt_tokens=num_prompt_tokens,
                num_generated_tokens=num_gen_tokens,
                **kv_cfg,
            )
        else:  # KIVI
            kv_stats = get_kivi_memory_stats(model)

        outputs.append(text)
        prompts.append(prompt)
        references.append(sample["highlights"])
        kv_theoretical.append(kv_stats)

        if i < num_print:
            print(
                f"[{label}] Example {i} | "
                f"KV theo: {kv_stats['total_mb']:.2f} MB | "
                f"Empirical peak GPU: {peak_mb:.2f} MB"
            )

        # 🔹 FULL CLEANUP AFTER SAMPLE
        del text
        del prompt_ids
        del gen_ids
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    return outputs, prompts, references, {
        "per_prompt_kv": kv_theoretical,
        "peak_gpu_mb_per_sample": peak_gpu_mb_per_sample,
    }

In [ ]:
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [ ]:
def main():
    model_name = "meta-llama/Llama-2-7b-chat-hf"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    cnn_dm = load_cnn_dm(split="validation", num_samples=100)
    articles = [sample["article"] for sample in cnn_dm]
    device="cuda"
    # ===============================
    # BASELINE PASS
    # ===============================
    baseline_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto" if device == "cuda" else None
    )
    if device != "cuda":
        baseline_model = baseline_model.to(device)

    baseline_outputs, prompts, references, baseline_mem = generate_all_summaries(
        baseline_model,
        tokenizer,
        cnn_dm,
        label="BASELINE",
        max_new_tokens=256,
    )

    del baseline_model
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    # ===============================
    # KIVI PASS
    # ===============================
    kivi_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map="auto" if device == "cuda" else None
    )
    if device != "cuda":
        kivi_model = kivi_model.to(device)
    kivi_model = replace_llama_attention_with_kivi(kivi_model)

    kivi_outputs, _, _, kivi_mem = generate_all_summaries(
        kivi_model,
        tokenizer,
        cnn_dm,
        label="KIVI",
        max_new_tokens=256,
    )

    del kivi_model
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    # ===============================
    # EVALUATION (CPU ONLY)
    # ===============================
    per_sample, results = evaluate_on_cnn_dm_from_texts(
        articles=articles,
        references=references,
        prompts=prompts,
        baseline_outputs=baseline_outputs,
        kivi_outputs=kivi_outputs,
        tokenizer=tokenizer,
    )

    memory_results = {
        "baseline": baseline_mem,
        "kivi": kivi_mem,
    }

    return per_sample, results, memory_results

if __name__ == "__main__":
    examples_cnn, results_cnn, memory_results_cnn = main()


In [ ]:
def print_example(ex, idx=None):
    print("=" * 100)
    if idx is not None:
        print(f"Example {idx}")
        print("-" * 100)

    print("PROMPT:\n", ex["prompt"])
    print("\nREFERENCE:\n", ex["reference"])
    print("\nBASELINE OUTPUT:\n", ex["baseline_output"])
    print("\nKIVI OUTPUT:\n", ex["kivi_output"])

    print("\nSCORES")
    for group, scores in ex["scores"].items():
        print(f"\n{group}")
        for k, v in scores.items():
            print(f"  {k}: {v:.4f}")


In [ ]:
print_example(examples_cnn[0], idx=0)


In [ ]:
def build_results_table(results, decimals=4):
    def fmt(x):
        if x == "—":
            return "—"
        if isinstance(x, dict):
            return f"{x['mean']:.{decimals}f} ± {x['std']:.{decimals}f}"
        if isinstance(x, float):
            return f"{x:.{decimals}f}"
        return str(x)

    table = {
        "ROUGE-L F1": {
            "Baseline vs GT": results["rouge_bg"],
            "KIVI vs GT": results["rouge_kg"],
            "Baseline vs KIVI": results["rouge_bk"],
        },
        "BERTScore F1": {
            "Baseline vs GT": results["bert_bg"],
            "KIVI vs GT": results["bert_kg"],
            "Baseline vs KIVI": results["bert_bk"],
        },
        "Token match rate": {
            "Baseline vs GT": "—",
            "KIVI vs GT": "—",
            "Baseline vs KIVI": results["token_bk"],
        },
        "Sentence error (SCE)": {
            "Baseline vs GT": results["sce_baseline"],
            "KIVI vs GT": results["sce_kivi"],
            "Baseline vs KIVI": "—",
        },
        "Sentence compliance (strict)": {
            "Baseline vs GT": results["scr_baseline"],
            "KIVI vs GT": results["scr_kivi"],
            "Baseline vs KIVI": "—",
        },
        # "NLI entailment": {
        #     # "Baseline vs GT": results["nli_ent_baseline"],
        #     # "KIVI vs GT": results["nli_ent_kivi"],
        #     "Baseline vs KIVI": "—",
        # },
        # # "NLI contradiction": {
        # #     # "Baseline vs GT": results["nli_con_baseline"],
        # #     # "KIVI vs GT": results["nli_con_kivi"],
        # #     "Baseline vs KIVI": "—",
        # },
        "Entity hallucination rate": {
            "Baseline vs GT": results["ent_hall_rate_baseline"],
            "KIVI vs GT": results["ent_hall_rate_kivi"],
            "Baseline vs KIVI": "—",
        },
    }

    # Format everything
    formatted = {}
    for metric, cols in table.items():
        formatted[metric] = {k: fmt(v) for k, v in cols.items()}

    return formatted

def print_results_table(table):
    header = (
        "Metric".ljust(30)
        + "Baseline vs GT".rjust(22)
        + "KIVI vs GT".rjust(22)
        + "Baseline vs KIVI".rjust(24)
    )
    print("\n" + header)
    print("-" * len(header))

    for metric, cols in table.items():
        print(
            metric.ljust(30)
            + cols["Baseline vs GT"].rjust(22)
            + cols["KIVI vs GT"].rjust(22)
            + cols["Baseline vs KIVI"].rjust(24)
        )



In [ ]:
table = build_results_table(results_cnn)
print_results_table(table)


In [ ]:
def print_per_prompt_kv(memory_results):
    baseline = memory_results["baseline"]["per_prompt_kv"]
    kivi = memory_results["kivi"]["per_prompt_kv"]

    print("\n📦 Per-Prompt Theoretical KV Cache Memory")
    print("=" * 70)
    print(f"{'Prompt':<8} {'Baseline KV (MB)':>20} {'KIVI KV (MB)':>20}")
    print("-" * 70)

    for i in range(len(baseline)):
        b_mb = baseline[i]["total_mb"]
        k_mb = kivi[i]["total_mb"]
        print(f"{i:<8} {b_mb:>20.2f} {k_mb:>20.2f}")


In [ ]:
print_per_prompt_kv(memory_results_cnn)

In [ ]:
import numpy as np

def print_aggregate_kv(memory_results):
    def summarize(per_prompt):
        mbs = [x["total_mb"] for x in per_prompt]
        return {
            "mean": np.mean(mbs),
            "max": np.max(mbs),
            "min": np.min(mbs),
        }

    base = summarize(memory_results["baseline"]["per_prompt_kv"])
    kivi = summarize(memory_results["kivi"]["per_prompt_kv"])

    compression = base["mean"] / kivi["mean"]

    print("\n📊 Aggregate Theoretical KV Cache (MB)")
    print("=" * 70)
    print(f"{'':<20} {'Baseline':>15} {'KIVI':>15}")
    print("-" * 70)
    print(f"{'Mean KV':<20} {base['mean']:>15.2f} {kivi['mean']:>15.2f}")
    print(f"{'Max KV':<20} {base['max']:>15.2f} {kivi['max']:>15.2f}")
    print(f"{'Min KV':<20} {base['min']:>15.2f} {kivi['min']:>15.2f}")
    print("-" * 70)
    print(f"Compression ratio (Baseline / KIVI): {compression:.2f}×")


In [ ]:
print_aggregate_kv(memory_results_cnn)

### Save Results to JSON


In [ ]:
import json
import numpy as np

def to_json_safe(obj):
    """
    Recursively convert numpy types to native Python types
    so json.dump won't crash.
    """
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

def save_cnn_results(
    examples,
    results,
    memory_results,
    out_dir="cnn_results",
):
    import os
    os.makedirs(out_dir, exist_ok=True)

    examples_path = f"{out_dir}/examples_cnn_100.json"
    results_path = f"{out_dir}/results_cnn_100.json"
    memory_path  = f"{out_dir}/memory_results_cnn_100.json"

    with open(examples_path, "w") as f:
        json.dump(to_json_safe(examples), f, indent=2)

    with open(results_path, "w") as f:
        json.dump(to_json_safe(results), f, indent=2)

    with open(memory_path, "w") as f:
        json.dump(to_json_safe(memory_results), f, indent=2)

    print("✅ Saved CNN results:")
    print(" -", examples_path)
    print(" -", results_path)
    print(" -", memory_path)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
save_cnn_results(
    examples=examples_cnn,
    results=results_cnn,
    memory_results=memory_results_cnn,
    out_dir="cnn_dm_kivi_eval",
)

In [ ]:
import numpy as np

def transform_memory_results_cnn(memory_results_cnn):
    """
    Transform memory_results_cnn (in-memory dict) into an explicit
    prompt-indexed KV schema.
    """

    baseline_kv = memory_results_cnn["baseline"]["per_prompt_kv"]
    kivi_kv = memory_results_cnn["kivi"]["per_prompt_kv"]

    assert len(baseline_kv) == len(kivi_kv), \
        "Mismatch: baseline and kivi prompt counts differ"

    prompts = []

    for prompt_id, (b, k) in enumerate(zip(baseline_kv, kivi_kv)):
        entry = {
            "prompt_id": prompt_id,

            "tokens": {
                "total": b.get("total_tokens", None),
            },

            "baseline": {
                "kv_mb": float(b["total_mb"]),
            },

            "kivi": {
                "kv_mb": float(k["total_mb"]),
            },
        }

        # Optional detailed KIVI breakdown (only if present)
        if "key_quant_bytes" in k:
            entry["kivi"]["breakdown"] = {
                "key_quant_mb": k.get("key_quant_bytes", 0) / (1024 ** 2),
                "value_quant_mb": k.get("value_quant_bytes", 0) / (1024 ** 2),
                "key_residual_mb": k.get("key_residual_bytes", 0) / (1024 ** 2),
                "value_residual_mb": k.get("value_residual_bytes", 0) / (1024 ** 2),
                "key_metadata_mb": k.get("key_metadata_bytes", 0) / (1024 ** 2),
                "value_metadata_mb": k.get("value_metadata_bytes", 0) / (1024 ** 2),
            }

        prompts.append(entry)

    baseline_mbs = [p["baseline"]["kv_mb"] for p in prompts]
    kivi_mbs = [p["kivi"]["kv_mb"] for p in prompts]

    transformed = {
        "metadata": {
            "measurement": "terminal_per_prompt_kv",
            "kv_unit": "MB",
            "dataset": "CNN/DailyMail",
            "notes": (
                "Each entry corresponds to KV cache memory after "
                "full autoregressive generation for one prompt."
            )
        },

        "prompts": prompts,

        "run_summary": {
            "baseline_peak_gpu_mb": memory_results_cnn["baseline"].get("peak_gpu_mb"),
            "kivi_peak_gpu_mb": memory_results_cnn["kivi"].get("peak_gpu_mb"),

            "mean_kv_mb": {
                "baseline": float(np.mean(baseline_mbs)),
                "kivi": float(np.mean(kivi_mbs)),
            },

            "compression_ratio_mean": float(
                np.mean(baseline_mbs) / np.mean(kivi_mbs)
            )
        }
    }

    return transformed


In [ ]:
promptwise_memory = transform_memory_results_cnn(memory_results_cnn)


# CoQA

In [ ]:
def build_coqa_prompt(context, question):
    return f"""<s>[INST]
Read the story and answer the question using a short phrase.

Story:
{context}

Question:
{question}
[/INST]"""


In [ ]:
@torch.no_grad()
def generate_coqa(model, tokenizer, prompt, max_new_tokens=32):
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)

    out = model.generate(
        **enc,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    gen = out[0][enc["input_ids"].shape[1]:]
    return tokenizer.decode(gen, skip_special_tokens=True).strip()


In [ ]:
import re
from collections import Counter

def normalize(text):
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = re.sub(r"[^a-z0-9 ]", "", text)
    return " ".join(text.split())

def coqa_f1(pred, ref):
    pred = normalize(pred)
    ref = normalize(ref)

    if pred == "" and ref == "":
        return 1.0
    if pred == "" or ref == "":
        return 0.0

    pred_tokens = pred.split()
    ref_tokens = ref.split()

    common = Counter(pred_tokens) & Counter(ref_tokens)
    num_same = sum(common.values())

    if num_same == 0:
        return 0.0

    precision = num_same / len(pred_tokens)
    recall = num_same / len(ref_tokens)
    return 2 * precision * recall / (precision + recall)

import re

def strip_answer_prefix(text):
    patterns = [
        r"^the answer to the question is[:\s]*",
        r"^answer[:\s]*",
    ]
    text = text.lower().strip()
    for p in patterns:
        text = re.sub(p, "", text)
    return text.strip()


def canonicalize_yes_no(text):
    text = text.lower()
    if re.search(r"\b(yes|yeah|yep)\b", text):
        return "yes"
    if re.search(r"\b(no|not|didn't|did not)\b", text):
        return "no"
    return text

def coqa_f1_robust(pred, ref):
    pred = strip_answer_prefix(pred)
    pred = canonicalize_yes_no(pred)
    ref  = canonicalize_yes_no(ref)

    return coqa_f1(pred, ref)

from rouge_score import rouge_scorer

def rouge_l_score(reference, prediction):
    scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
    scores = scorer.score(reference, prediction)
    return scores["rougeL"].fmeasure

from bert_score import score as bertscore

def bert_score_f1(
    reference_text,
    prediction_text,
    model_type="roberta-large",
    device="cpu",
):
    P, R, F1 = bertscore(
        [prediction_text],
        [reference_text],
        lang="en",
        model_type=model_type,
        device=device,
        verbose=False,
    )
    return F1.mean().item()

def token_level_match_rate(
    baseline_text,
    kivi_text,
    tokenizer,
):
    base_tokens = tokenizer.encode(baseline_text, add_special_tokens=False)
    kivi_tokens = tokenizer.encode(kivi_text, add_special_tokens=False)

    min_len = min(len(base_tokens), len(kivi_tokens))

    matches = sum(
        base_tokens[i] == kivi_tokens[i]
        for i in range(min_len)
    )

    return matches / min_len if min_len > 0 else 0.0

In [ ]:
from datasets import load_dataset

def load_coqa(n=200, split="validation"):
    ds = load_dataset("coqa", split=split)
    data = []

    for ex in ds:
        context = ex["story"]
        for i in range(len(ex["questions"])):
            data.append({
                "context": context,
                "question": ex["questions"][i],
                "answer": ex["answers"]["input_text"][i],
            })
            if len(data) >= n:
                return data
    return data


In [ ]:
import gc
import torch
from transformers import AutoTokenizer, LlamaForCausalLM

def load_models():
    model_name = "meta-llama/Llama-2-7b-chat-hf"
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)

    # 🔴 critical for chat models
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    # ===== BASELINE =====
    baseline = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    # Free memory before KIVI
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    # ===== KIVI =====
    kivi = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    kivi = replace_llama_attention_with_kivi(kivi)

    return tokenizer, baseline, kivi


In [ ]:
@torch.no_grad()
def evaluate_single_model(
    model,
    tokenizer,
    data,
    debug_n=3,
    max_new_tokens=32,
):
    f1_scores = []

    for idx, ex in enumerate(data):
        prompt = build_coqa_prompt(ex["context"], ex["question"])

        output = generate_coqa(
            model,
            tokenizer,
            prompt,
            max_new_tokens=max_new_tokens,
        )

        score = coqa_f1(output, ex["answer"])
        f1_scores.append(score)

        # 🔍 Debug prints (first few examples)
        if idx < debug_n:
            print("=" * 80)
            print(f"Example {idx}")
            print("- Prompt:")
            print(prompt)
            print("- Ground Truth:", ex["answer"])
            print("- Model Output:", output)
            print(f"- CoQA F1: {score:.3f}")

    return sum(f1_scores) / len(f1_scores)

from collections import defaultdict

def evaluate_all_metrics(
    baseline_outputs,
    kivi_outputs,
    ground_truths,
    tokenizer,
    device="cpu",
):
    per_example = []
    agg = defaultdict(list)

    for base_out, kivi_out, gt in zip(
        baseline_outputs, kivi_outputs, ground_truths
    ):
        ex_scores = {
            "baseline_vs_gt": {
                "f1_raw": coqa_f1(base_out, gt),
                "f1_robust": coqa_f1_robust(base_out, gt),
                "rouge_l": rouge_l_score(gt, base_out),
                "bert": bert_score_f1(gt, base_out, device=device),
            },
            "kivi_vs_gt": {
                "f1_raw": coqa_f1(kivi_out, gt),
                "f1_robust": coqa_f1_robust(kivi_out, gt),
                "rouge_l": rouge_l_score(gt, kivi_out),
                "bert": bert_score_f1(gt, kivi_out, device=device),
            },
            "baseline_vs_kivi": {
                "token_match": token_level_match_rate(
                    base_out, kivi_out, tokenizer
                ),
                "rouge_l": rouge_l_score(base_out, kivi_out),
                "bert": bert_score_f1(base_out, kivi_out, device=device),
            },
        }

        # collect for aggregation
        for side in ["baseline_vs_gt", "kivi_vs_gt"]:
            agg[f"{side}_f1_raw"].append(ex_scores[side]["f1_raw"])
            agg[f"{side}_f1_robust"].append(ex_scores[side]["f1_robust"])
            agg[f"{side}_rouge"].append(ex_scores[side]["rouge_l"])
            agg[f"{side}_bert"].append(ex_scores[side]["bert"])

        agg["bk_token_match"].append(
            ex_scores["baseline_vs_kivi"]["token_match"]
        )
        agg["bk_rouge"].append(
            ex_scores["baseline_vs_kivi"]["rouge_l"]
        )
        agg["bk_bert"].append(
            ex_scores["baseline_vs_kivi"]["bert"]
        )

        per_example.append(ex_scores)

    aggregate = {k: sum(v) / len(v) for k, v in agg.items()}

    return {
        "aggregate": aggregate,
        "per_example": per_example,
    }

def build_results_table(m):
    return {
        "Token F1 (raw)": {
            "Baseline → GT": round(m["baseline_vs_gt_f1_raw"], 3),
            "KIVI → GT": round(m["kivi_vs_gt_f1_raw"], 3),
            "Baseline ↔ KIVI": "—",
        },
        "Token F1 (robust)": {
            "Baseline → GT": round(m["baseline_vs_gt_f1_robust"], 3),
            "KIVI → GT": round(m["kivi_vs_gt_f1_robust"], 3),
            "Baseline ↔ KIVI": "—",
        },
        "ROUGE-L": {
            "Baseline → GT": round(m["baseline_vs_gt_rouge"], 3),
            "KIVI → GT": round(m["kivi_vs_gt_rouge"], 3),
            "Baseline ↔ KIVI": round(m["bk_rouge"], 3),
        },
        "BERTScore": {
            "Baseline → GT": round(m["baseline_vs_gt_bert"], 3),
            "KIVI → GT": round(m["kivi_vs_gt_bert"], 3),
            "Baseline ↔ KIVI": round(m["bk_bert"], 3),
        },
        "Token match rate": {
            "Baseline → GT": "—",
            "KIVI → GT": "—",
            "Baseline ↔ KIVI": round(m["bk_token_match"], 3),
        },
    }

In [ ]:
def evaluate_coqa(baseline, kivi, tokenizer, data, debug_n=3):
    baseline_f1 = []
    kivi_f1 = []

    for idx, ex in enumerate(data):
        prompt = build_coqa_prompt(ex["context"], ex["question"])

        base_out = generate_coqa(baseline, tokenizer, prompt)
        kivi_out = generate_coqa(kivi, tokenizer, prompt)

        base_score = coqa_f1(base_out, ex["answer"])
        kivi_score = coqa_f1(kivi_out, ex["answer"])

        baseline_f1.append(base_score)
        kivi_f1.append(kivi_score)

        # 🔍 DEBUG PRINT
        if idx < debug_n:
            print("=" * 80)
            print(f"Example {idx}")
            print("- Prompt:")
            print(prompt)
            print("- GT Answer:", ex["answer"])
            print("- Baseline:", base_out)
            print("- KIVI:", kivi_out)
            print(f"- F1 (Baseline / KIVI): {base_score:.3f} / {kivi_score:.3f}")

    return sum(baseline_f1) / len(baseline_f1), sum(kivi_f1) / len(kivi_f1)

@torch.no_grad()
def run_model_and_collect_outputs_with_kv(
    model,
    tokenizer,
    data,
    label,
    max_new_tokens=32,s
    debug_n=3,
):
    outputs = []
    ground_truths = []
    kv_theoretical = []
    peak_gpu_mb_per_prompt = []

    kv_cfg = get_model_kv_config(model)

    for idx, ex in enumerate(data):
        prompt = build_coqa_prompt(ex["context"], ex["question"])

        # 🔹 HARD RESET — isolate THIS prompt only
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

        # ---- TOKEN COUNTS (prompt) ----
        prompt_ids = tokenizer(
            prompt, return_tensors="pt", add_special_tokens=False
        )["input_ids"]
        num_prompt_tokens = prompt_ids.shape[-1]

        # ---- GENERATION ----
        out = generate_coqa(
            model,
            tokenizer,
            prompt,
            max_new_tokens=max_new_tokens,
        )

        # 🔹 EMPIRICAL PEAK GPU MEMORY (THIS PROMPT)
        peak_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
        peak_gpu_mb_per_prompt.append(peak_mb)

        # ---- TOKEN COUNTS (generated) ----
        gen_ids = tokenizer(
            out, return_tensors="pt", add_special_tokens=False
        )["input_ids"]
        num_gen_tokens = gen_ids.shape[-1]

        # ---- THEORETICAL KV ----
        if label == "BASELINE":
            kv_stats = theoretical_baseline_kv_mb(
                num_prompt_tokens=num_prompt_tokens,
                num_generated_tokens=num_gen_tokens,
                **kv_cfg,
            )
        else:  # KIVI
            kv_stats = get_kivi_memory_stats(model)

        outputs.append(out)
        ground_truths.append(ex["answer"])
        kv_theoretical.append(kv_stats)

        if idx < debug_n:
            print("=" * 80)
            print(f"[{label}] Example {idx}")
            print("- Prompt:\n", prompt)
            print("- GT:", ex["answer"])
            print("- Output:", out)
            print(f"- KV theoretical (MB): {kv_stats['total_mb']:.2f}")
            print(f"- Empirical GPU peak (MB): {peak_mb:.2f}")

        # 🔹 CLEANUP (important for correctness)
        del out, prompt_ids, gen_ids
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    return outputs, ground_truths, {
        "per_prompt_kv": kv_theoretical,
        "per_prompt_empirical_gpu_mb": peak_gpu_mb_per_prompt,
        "theoretical_peak_kv_mb": max(kv["total_mb"] for kv in kv_theoretical),
        "empirical_peak_gpu_mb": max(peak_gpu_mb_per_prompt),
    }

In [ ]:
def main():
    model_name = "meta-llama/Llama-2-7b-chat-hf"

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        use_fast=False
    )
    tokenizer.pad_token = tokenizer.eos_token

    data = load_coqa(n=)

    # =====================================================
    # BASELINE
    # =====================================================
    baseline = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

    baseline_outputs, ground_truths, baseline_mem = (
        run_model_and_collect_outputs_with_kv(
            baseline,
            tokenizer,
            data,
            label="BASELINE",
            debug_n=3,
        )
    )

    del baseline
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    # =====================================================
    # KIVI
    # =====================================================
    kivi = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    kivi = replace_llama_attention_with_kivi(kivi)

    kivi_outputs, _, kivi_mem = (
        run_model_and_collect_outputs_with_kv(
            kivi,
            tokenizer,
            data,
            label="KIVI",
            debug_n=3,
        )
    )


    del kivi
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    # =====================================================
    # METRICS (CPU ONLY)
    # =====================================================
    results = evaluate_all_metrics(
        baseline_outputs=baseline_outputs,
        kivi_outputs=kivi_outputs,
        ground_truths=ground_truths,
        tokenizer=tokenizer,
        device="cpu",
    )

    # =====================================================
    # BUILD PER-EXAMPLE RECORDS (SAVE FOR LATER)
    # =====================================================
    examples = []
    for i in range(len(ground_truths)):
        examples.append({
            "prompt": build_coqa_prompt(
                data[i]["context"],
                data[i]["question"],
            ),
            "ground_truth": ground_truths[i],
            "baseline_output": baseline_outputs[i],
            "kivi_output": kivi_outputs[i],
            "scores": results["per_example"][i],
        })

    # =====================================================
    # AGGREGATE RESULTS TABLE
    # =====================================================
    table = build_results_table(results["aggregate"])

    print("\n📊 FINAL RESULTS")
    print(
        "Metric".ljust(18),
        "Baseline→GT".rjust(14),
        "KIVI→GT".rjust(12),
        "Baseline↔KIVI".rjust(18),
    )
    for metric, cols in table.items():
        print(
            metric.ljust(18),
            f"{cols['Baseline → GT']}".rjust(14),
            f"{cols['KIVI → GT']}".rjust(12),
            f"{cols['Baseline ↔ KIVI']}".rjust(18),
        )


    memory_results = {
        "baseline": baseline_mem,
        "kivi": kivi_mem,
    }

    return examples, results["aggregate"], memory_results


if __name__ == "__main__":
    examples, aggregate_metrics, memory_results = main()

In [ ]:
memory_results

In [ ]:
import json
import numpy as np

def to_json_safe(obj):
    """
    Recursively convert numpy types to native Python types
    so json.dump won't crash.
    """
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    else:
        return obj

def save_cnn_results(
    examples,
    results,
    memory_results,
    out_dir="cnn_results",
):
    import os
    os.makedirs(out_dir, exist_ok=True)

    examples_path = f"{out_dir}/examples_coqa-500.json"
    results_path = f"{out_dir}/results_coqa-500.json"
    memory_path  = f"{out_dir}/memory_results_coqa-500.json"

    with open(examples_path, "w") as f:
        json.dump(to_json_safe(examples), f, indent=2)

    with open(results_path, "w") as f:
        json.dump(to_json_safe(results), f, indent=2)

    with open(memory_path, "w") as f:
        json.dump(to_json_safe(memory_results), f, indent=2)

    print("✅ Saved CNN results:")
    print(" -", examples_path)
    print(" -", results_path)
    print(" -", memory_path)


In [ ]:
save_cnn_results(
  examples=examples,
  results=aggregate_metrics,
  memory_results=memory_results,
    out_dir="coqa_dm_kivi_eval",
)

In [ ]:
def print_example(ex, idx=None):
    print("=" * 100)
    if idx is not None:
        print(f"Example {idx}")
        print("-" * 100)

    print("PROMPT:")
    print(ex["prompt"])
    print()

    print("GROUND TRUTH:")
    print(ex["ground_truth"])
    print()

    print("BASELINE OUTPUT:")
    print(ex["baseline_output"])
    print()

    print("KIVI OUTPUT:")
    print(ex["kivi_output"])
    print()

    print("SCORES:")
    for pair, scores in ex["scores"].items():
        print(f"  {pair}:")
        for k, v in scores.items():
            print(f"    {k}: {v:.4f}")


In [ ]:
print_example(examples[0], idx=0)

In [ ]:
print_example(examples[1], idx=1)

In [ ]:
print_example(examples[0], idx=0)

In [ ]:
def build_coqa_aggregate_table(agg, decimals=3):
    def fmt(x):
        return f"{x:.{decimals}f}"

    return {
        "Token match rate": {
            "Baseline → GT": "—",
            "KIVI → GT": "—",
            "Baseline ↔ KIVI": fmt(agg["bk_token_match"]),
        },
        "Token F1 (raw)": {
            "Baseline → GT": fmt(agg["baseline_vs_gt_f1_raw"]),
            "KIVI → GT": fmt(agg["kivi_vs_gt_f1_raw"]),
            "Baseline ↔ KIVI": "—",
        },
        "Token F1 (robust)": {
            "Baseline → GT": fmt(agg["baseline_vs_gt_f1_robust"]),
            "KIVI → GT": fmt(agg["kivi_vs_gt_f1_robust"]),
            "Baseline ↔ KIVI": "—",
        },
        "ROUGE-L": {
            "Baseline → GT": fmt(agg["baseline_vs_gt_rouge"]),
            "KIVI → GT": fmt(agg["kivi_vs_gt_rouge"]),
            "Baseline ↔ KIVI": fmt(agg["bk_rouge"]),
        },
        "BERTScore": {
            "Baseline → GT": fmt(agg["baseline_vs_gt_bert"]),
            "KIVI → GT": fmt(agg["kivi_vs_gt_bert"]),
            "Baseline ↔ KIVI": fmt(agg["bk_bert"]),
        },
    }
def print_coqa_aggregate_table(table):
    print("\n📊 FINAL RESULTS (CoQA)")
    print(
        "Metric".ljust(22),
        "Baseline→GT".rjust(14),
        "KIVI→GT".rjust(12),
        "Baseline↔KIVI".rjust(18),
    )
    print("-" * 68)

    for metric, cols in table.items():
        print(
            metric.ljust(22),
            cols["Baseline → GT"].rjust(14),
            cols["KIVI → GT"].rjust(12),
            cols["Baseline ↔ KIVI"].rjust(18),
        )


In [ ]:
table = build_coqa_aggregate_table(aggregate_metrics)
print_coqa_aggregate_table(table)

In [ ]:
aggregate_metrics


In [ ]:
memory_results


# GSM8k dataset

In [ ]:
from datasets import load_dataset

def load_gsm8k(split="test", num_samples=None):
    """
    Load GSM8K dataset.
    split: 'train' or 'test'
    num_samples: optional int to subsample
    """
    ds = load_dataset("gsm8k", "main", split=split)

    if num_samples is not None:
        ds = ds.select(range(num_samples))

    return ds


In [ ]:
ds=load_gsm8k(split="test", num_samples=2)
sample = ds[0]
print(sample.keys())
print(sample["question"])
print(sample["answer"])


In [ ]:
import re

def extract_final_answer(text):
    """
    Extract final numeric answer from GSM8K-style output.
    Returns None if not found.
    """
    matches = re.findall(r"-?\d+\.?\d*", text.replace(",", ""))
    return matches[-1] if matches else None

def build_gsm8k_prompt(question):
    return f"""Solve the following math problem step by step.
Provide the final answer as a number.

Question:
{question}

Answer:
"""
def generate_gsm8k_answer(
    model,
    tokenizer,
    prompt,
    label,
    max_new_tokens=256,
    verbose=False,
):
    if verbose:
        print(f"\n=========== {label} PROMPT ===========")
        print(prompt)
        print("=====================================")

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs["input_ids"].shape[-1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    gen_ids = outputs[0][prompt_len:]
    text = tokenizer.decode(gen_ids, skip_special_tokens=True)

    if verbose:
        print(f"\n=========== {label} OUTPUT ===========")
        print(text)
        print("=====================================")

    return text

def generate_all_gsm8k_answers(
    model,
    tokenizer,
    dataset,
    label,
    max_new_tokens=256,
    num_print=3,
):
    outputs = []
    prompts = []
    gt_answers = []
    kv_theoretical = []

    kv_cfg = get_model_kv_config(model)

    # 🔹 Full-run peak GPU memory
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    for i, sample in enumerate(dataset):
        prompt = build_gsm8k_prompt(sample["question"])

        prompt_ids = tokenizer(
            prompt, return_tensors="pt", add_special_tokens=False
        )["input_ids"]
        num_prompt_tokens = prompt_ids.shape[-1]

        text = generate_gsm8k_answer(
            model,
            tokenizer,
            prompt,
            label,
            max_new_tokens=max_new_tokens,
            verbose=(i < num_print),
        )

        gen_ids = tokenizer(
            text, return_tensors="pt", add_special_tokens=False
        )["input_ids"]
        num_gen_tokens = gen_ids.shape[-1]

        # ---- THEORETICAL KV ----
        if label == "BASELINE":
            kv_stats = theoretical_baseline_kv_mb(
                num_prompt_tokens=num_prompt_tokens,
                num_generated_tokens=num_gen_tokens,
                **kv_cfg,
            )
        else:  # KIVI
            kv_stats = get_kivi_memory_stats(model)

        outputs.append(text)
        prompts.append(prompt)
        gt_answers.append(extract_final_answer(sample["answer"]))
        kv_theoretical.append(kv_stats)

    peak_gpu_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)

    return outputs, prompts, gt_answers, {
        "per_prompt_kv": kv_theoretical,
        "peak_gpu_mb": peak_gpu_mb,
    }


In [ ]:
import numpy as np

def evaluate_on_gsm8k(
    dataset,
    tokenizer,
    baseline_model,
    kivi_model,
    max_new_tokens=256,
    num_print=3,
):
    stats = {
        "baseline_correct": [],
        "kivi_correct": [],
        "consistency_rate": [],
        "baseline_to_kivi_fail": [],
        "token_match": [],
    }
    per_sample = []
    for i, sample in enumerate(dataset):
        verbose = i < num_print
        if verbose:
            print(f"\n================ Example {i} =================")

        prompt = build_gsm8k_prompt(sample["question"])
        gt_answer = extract_final_answer(sample["answer"])

        baseline_text = generate_gsm8k_answer(
            baseline_model,
            tokenizer,
            prompt,
            "BASELINE",
            max_new_tokens,
            verbose,
        )
        kivi_text = generate_gsm8k_answer(
            kivi_model,
            tokenizer,
            prompt,
            "KIVI",
            max_new_tokens,
            verbose,
        )

        base_ans = extract_final_answer(baseline_text)
        kivi_ans = extract_final_answer(kivi_text)

        base_correct = int(base_ans == gt_answer)
        kivi_correct = int(kivi_ans == gt_answer)

        consistency = int(base_correct == kivi_correct)
        base_to_kivi_fail = int(base_correct == 1 and kivi_correct == 0)

        token_stats = token_level_match_rate(
            baseline_text, kivi_text, tokenizer
        )
        per_sample.append({
            "idx": i,

            "baseline_correct": base_correct,
            "kivi_correct": kivi_correct,

            "token_match_rate": token_stats["token_match_rate"],

            "baseline_to_kivi_fail": base_correct and not kivi_correct,
            "both_correct": base_correct and kivi_correct,
        })
        # store
        stats["baseline_correct"].append(base_correct)
        stats["kivi_correct"].append(kivi_correct)
        stats["consistency_rate"].append(consistency)
        stats["baseline_to_kivi_fail"].append(base_to_kivi_fail)
        stats["token_match"].append(token_stats["token_match_rate"])

        if verbose:
            print(f"\n=== GSM8K Example {i} ===")
            print(f"GT answer:        {gt_answer}")
            print(f"Baseline answer:  {base_ans} ({'✓' if base_correct else '✗'})")
            print(f"KIVI answer:      {kivi_ans} ({'✓' if kivi_correct else '✗'})")
            print(f"Token match rate: {token_stats['token_match_rate']:.3f}")

    return  per_sample, {
        "baseline_EM": np.mean(stats["baseline_correct"]),
        "kivi_EM": np.mean(stats["kivi_correct"]),
        "consistency": np.mean(stats["consistency_rate"]),
        "baseline_to_kivi_fail": np.mean(stats["baseline_to_kivi_fail"]),
        "token_match": {
            "mean": np.mean(stats["token_match"]),
            "std":  np.std(stats["token_match"]),
        }
    }

def evaluate_on_gsm8k_from_texts(
    prompts,
    gt_answers,
    baseline_outputs,
    kivi_outputs,
    tokenizer,
):
    stats = {
        "baseline_correct": [],
        "kivi_correct": [],
        "baseline_to_kivi_fail": [],
        "consistency": [],
        "token_match": [],
    }

    per_sample = []

    for i in range(len(prompts)):
        gt = gt_answers[i]

        base_text = baseline_outputs[i]
        kivi_text = kivi_outputs[i]

        base_ans = extract_final_answer(base_text)
        kivi_ans = extract_final_answer(kivi_text)

        base_correct = int(base_ans == gt)
        kivi_correct = int(kivi_ans == gt)

        token_stats = token_level_match_rate(base_text, kivi_text, tokenizer)

        row = {
            "idx": i,

            # ---- TEXT ----
            "prompt": prompts[i],
            "ground_truth": gt,
            "baseline_output": base_text,
            "kivi_output": kivi_text,

            # ---- SCORES ----
            "scores": {
                "baseline_correct": base_correct,
                "kivi_correct": kivi_correct,
                "baseline_to_kivi_fail": int(base_correct and not kivi_correct),
                "consistency": int(base_correct == kivi_correct),
                "token_match_rate": token_stats["token_match_rate"],
            },
        }

        per_sample.append(row)

        # aggregate
        stats["baseline_correct"].append(base_correct)
        stats["kivi_correct"].append(kivi_correct)
        stats["baseline_to_kivi_fail"].append(base_correct and not kivi_correct)
        stats["consistency"].append(base_correct == kivi_correct)
        stats["token_match"].append(token_stats["token_match_rate"])

    aggregate = {
        "baseline_EM": np.mean(stats["baseline_correct"]),
        "kivi_EM": np.mean(stats["kivi_correct"]),
        "consistency": np.mean(stats["consistency"]),
        "baseline_to_kivi_fail": np.mean(stats["baseline_to_kivi_fail"]),
        "token_match": {
            "mean": np.mean(stats["token_match"]),
            "std": np.std(stats["token_match"]),
        },
    }

    return per_sample, aggregate


import matplotlib.pyplot as plt

def plot_token_match_rate(per_sample):
    x = [s["idx"] for s in per_sample]
    y = [s["token_match_rate"] for s in per_sample]

    plt.figure(figsize=(8, 4))
    plt.plot(x, y, marker="o")
    plt.xlabel("Sample index")
    plt.ylabel("Token match rate")
    plt.title("Baseline vs KIVI token match rate (per sample)")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.show()

def plot_correctness(per_sample):
    x = [s["idx"] for s in per_sample]
    base = [s["baseline_correct"] for s in per_sample]
    kivi = [s["kivi_correct"] for s in per_sample]

    plt.figure(figsize=(8, 4))
    plt.scatter(x, base, label="Baseline correct", marker="o")
    plt.scatter(x, kivi, label="KIVI correct", marker="x")
    plt.yticks([0, 1], ["Incorrect", "Correct"])
    plt.xlabel("Sample index")
    plt.ylabel("Correctness")
    plt.title("Correctness vs Ground Truth (GSM8K)")
    plt.legend()
    plt.grid(True)
    plt.show()

def plot_baseline_to_kivi_failures(per_sample):
    x = [s["idx"] for s in per_sample]
    failures = [int(s["baseline_to_kivi_fail"]) for s in per_sample]

    plt.figure(figsize=(8, 3))
    plt.bar(x, failures)
    plt.xlabel("Sample index")
    plt.ylabel("Failure (1 = yes)")
    plt.title("Baseline ✓ → KIVI ✗ failures")
    plt.ylim(0, 1.2)
    plt.grid(axis="y")
    plt.show()

def plot_token_match_vs_correctness(per_sample):
    token = [s["token_match_rate"] for s in per_sample]
    correct = [s["kivi_correct"] for s in per_sample]

    plt.figure(figsize=(5, 4))
    plt.scatter(token, correct)
    plt.xlabel("Token match rate (Baseline vs KIVI)")
    plt.ylabel("KIVI correctness")
    plt.yticks([0, 1], ["Incorrect", "Correct"])
    plt.title("Stability vs correctness")
    plt.grid(True)
    plt.show()


In [ ]:
def main():
    model_name = "meta-llama/Llama-2-13b-chat-hf"
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    gsm8k = load_gsm8k(split="test", num_samples=50)

    # ========= BASELINE =========
    baseline_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )

    baseline_outputs, prompts, gt_answers, baseline_mem = generate_all_gsm8k_answers(
        baseline_model, tokenizer, gsm8k, label="BASELINE"
    )

    del baseline_model
    torch.cuda.empty_cache()

    # ========= KIVI =========
    kivi_model = LlamaForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float16,
        device_map="auto",
    )
    kivi_model = replace_llama_attention_with_kivi(kivi_model)

    kivi_outputs, _, _, kivi_mem = generate_all_gsm8k_answers(
        kivi_model, tokenizer, gsm8k, label="KIVI"
    )

    del kivi_model
    torch.cuda.empty_cache()

    # ========= EVALUATION =========
    per_sample, aggregate = evaluate_on_gsm8k_from_texts(
        prompts=prompts,
        gt_answers=gt_answers,
        baseline_outputs=baseline_outputs,
        kivi_outputs=kivi_outputs,
        tokenizer=tokenizer,
    )

    memory = {
        "baseline": baseline_mem,
        "kivi": kivi_mem,
    }
    return per_sample, aggregate, memory

if __name__ == "__main__":
    examples, aggregate , memory_results= main()


In [ ]:
def build_gsm8k_aggregate_table(results):
    def f(x):
        return float(x)

    return {
        "Baseline EM": f"{f(results['baseline_EM']):.2f}",
        "KIVI EM": f"{f(results['kivi_EM']):.2f}",
        "Consistency rate": f"{f(results['consistency']):.2f}",
        "Baseline → KIVI failure": f"{f(results['baseline_to_kivi_fail']):.2f}",
        "Token match rate": (
            f"{f(results['token_match']['mean']):.2f} ± "
            f"{f(results['token_match']['std']):.2f}"
        ),
    }

def print_gsm8k_aggregate_table(table):
    print("\n📊 GSM8K AGGREGATE RESULTS")
    print(f"{'Metric':<30} {'Value':>15}")
    print("-" * 47)

    for metric, value in table.items():
        print(f"{metric:<30} {value:>15}")


In [ ]:
table = build_gsm8k_aggregate_table(aggregate)
print_gsm8k_aggregate_table(table)


In [ ]:
def print_example(ex, idx=None):
    header = f"Example {idx}" if idx is not None else "Example"
    print("\n" + "=" * 80)
    print(header)
    print("-" * 80)

    print("PROMPT:\n")
    print(ex["prompt"])

    if "ground_truth" in ex:
        print("\nGROUND TRUTH:\n")
        print(ex["ground_truth"])

    print("\nBASELINE OUTPUT:\n")
    print(ex["baseline_output"])

    print("\nKIVI OUTPUT:\n")
    print(ex["kivi_output"])

    print("\nSCORES:\n")

    for key, value in ex["scores"].items():
        if isinstance(value, dict):
            print(f"  {key}:")
            for k, v in value.items():
                if isinstance(v, float):
                    print(f"    {k}: {v:.4f}")
                else:
                    print(f"    {k}: {v}")
        else:
            if isinstance(value, float):
                print(f"  {key}: {value:.4f}")
            else:
                print(f"  {key}: {value}")


In [ ]:
ex = examples[2]
print_example(ex, idx=2)


In [ ]:
memory_results

In [ ]:
def print_per_prompt_kv(memory_results):
    baseline = memory_results["baseline"]["per_prompt_kv"]
    kivi = memory_results["kivi"]["per_prompt_kv"]

    print("\n📦 Per-Prompt Theoretical KV Cache Memory")
    print("=" * 70)
    print(f"{'Prompt':<8} {'Baseline KV (MB)':>20} {'KIVI KV (MB)':>20}")
    print("-" * 70)

    for i in range(len(baseline)):
        b_mb = baseline[i]["total_mb"]
        k_mb = kivi[i]["total_mb"]
        print(f"{i:<8} {b_mb:>20.2f} {k_mb:>20.2f}")


In [ ]:
print_per_prompt_kv(memory_results)

In [ ]:
import numpy as np

def print_aggregate_kv(memory_results):
    def summarize(per_prompt):
        mbs = [x["total_mb"] for x in per_prompt]
        return {
            "mean": np.mean(mbs),
            "max": np.max(mbs),
            "min": np.min(mbs),
        }

    base = summarize(memory_results["baseline"]["per_prompt_kv"])
    kivi = summarize(memory_results["kivi"]["per_prompt_kv"])

    compression = base["mean"] / kivi["mean"]

    print("\n📊 Aggregate Theoretical KV Cache (MB)")
    print("=" * 70)
    print(f"{'':<20} {'Baseline':>15} {'KIVI':>15}")
    print("-" * 70)
    print(f"{'Mean KV':<20} {base['mean']:>15.2f} {kivi['mean']:>15.2f}")
    print(f"{'Max KV':<20} {base['max']:>15.2f} {kivi['max']:>15.2f}")
    print(f"{'Min KV':<20} {base['min']:>15.2f} {kivi['min']:>15.2f}")
    print("-" * 70)
    print(f"Compression ratio (Baseline / KIVI): {compression:.2f}×")


In [ ]:
print_aggregate_kv(memory_results)

In [ ]:
def print_full_run_gpu_memory(memory_results):
    print("\n🔥 Full-Run Peak GPU Memory (empirical)")
    print("=" * 70)
    print(f"Baseline peak GPU memory: {memory_results['baseline']['peak_gpu_mb']:.2f} MB")
    print(f"KIVI peak GPU memory:     {memory_results['kivi']['peak_gpu_mb']:.2f} MB")


In [ ]:
print_full_run_gpu_memory(memory_results)

In [ ]:
def print_all_memory_results(memory_results):
    print_per_prompt_kv(memory_results)
    print_aggregate_kv(memory_results)
    print_full_run_gpu_memory(memory_results)


In [ ]:
print_all_memory_results(memory_results)

In [ ]:
memory_results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import json
from pathlib import Path

def save_gsm8k_results_json(
    per_sample,
    path="gsm8k_full_vs_sliding_vs_streaming.json",
):
    """
    Save GSM8K per-sample results (Full / Sliding / Streaming) to JSON.
    """

    # Ensure JSON-safe types
    data = []
    for s in per_sample:
        data.append({
            "idx": int(s["idx"]),
            "gt": s["gt"],

            "full": {
                "answer": s["full_ans"],
                "correct": int(s["full_correct"]),
            },
            "sliding": {
                "answer": s["slide_ans"],
                "correct": int(s["slide_correct"]),
            },
            "streaming": {
                "answer": s["stream_ans"],
                "correct": int(s["stream_correct"]),
            },
        })

    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with open(path, "w") as f:
        json.dump(data, f, indent=2)

    print(f"✅ Saved GSM8K results to {path}")


In [ ]:
examples

In [ ]:
aggregate

In [ ]:
memory_results

In [ ]:
import os, json
import numpy as np

def to_json_safe(obj):
    if isinstance(obj, dict):
        return {k: to_json_safe(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_json_safe(v) for v in obj]
    if isinstance(obj, tuple):
        return [to_json_safe(v) for v in obj]   # tuples -> lists
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

def save_results(examples, results, memory_results, out_dir="results", prefix="gsm8k"):
    os.makedirs(out_dir, exist_ok=True)

    examples_path = f"{out_dir}/examples_{prefix}.json"
    results_path  = f"{out_dir}/results_{prefix}.json"
    memory_path   = f"{out_dir}/memory_{prefix}.json"

    with open(examples_path, "w", encoding="utf-8") as f:
        json.dump(to_json_safe(examples), f, indent=2, ensure_ascii=False)

    with open(results_path, "w", encoding="utf-8") as f:
        json.dump(to_json_safe(results), f, indent=2, ensure_ascii=False)

    with open(memory_path, "w", encoding="utf-8") as f:
        json.dump(to_json_safe(memory_results), f, indent=2, ensure_ascii=False)

    print("✅ Saved results:")
    print(" -", examples_path)
    print(" -", results_path)
    print(" -", memory_path)

# usage
save_results(
    examples=examples,
    results=aggregate,
    memory_results=memory_results,
    out_dir="gsm8k_13b_dm_kivi_eval",
    prefix="gsm8k",
)
